# One-Phonon Diffuse Scattering — Fit to Experimental Data (6o2h, P1)

One rigid body per unit cell (the lysozyme molecule), 6 degrees of freedom per
cell: a rotation $\Omega$ and a translation $v$. The crystal's phonons are the
normal modes of a Born–von Kármán lattice of these rigid bodies, coupled through
a small set of pairwise contact springs. This notebook fits the contact
stiffnesses directly to the **experimental** diffuse-scattering maps deposited at
[CXIDB ID 128](https://www.cxidb.org/id-128.html)
(`triclinic_lysozyme_maps.h5`), associated with

> Meisburger, S. P., Case, D. A. & Ando, N. *Diffuse X-ray Scattering from
> Correlated Motions in a Protein Crystal.* Nat. Commun. 11, 1271 (2020).

**Pipeline:** deposited structure + experimental map metadata → atomic model and
mass matrix → molecular transform and $G(\mathbf q)$ → contact detection,
framing, and the geometric prior → dynamical matrix → halo-profile and
stratified mid-zone sampling of the experimental map → refinement of the contact
stiffness *shape* with the intensity scale profiled → sloppiness analysis →
predicted vs. deposited atomic displacement parameters → mode animations.

**What counts as validation here.** The deposited 6o2h ADPs are the independent
check, so nothing in the refinement is allowed to use them. In particular there
is **no** post-hoc rescaling of $K$ to make the predicted mean $B$ match the
deposited mean: that would make the headline comparison a tautology, and it is
also guaranteed to over-soften $K$, since the deposited $B$ contains internal and
substitutional disorder that a rigid-body lattice model cannot and should not
reproduce. The predicted mean $B$ should come out **below** the deposited mean,
and how far below is a result, not a defect.

The $K$/scale degeneracy is instead removed by construction (the refinement runs
on the normalized *shape* of $K$) and closed with **units**: the deposited map is
in electron scattering per unit cell, and $I=G^\dagger D^{-1}G$ with $k_BT=1$ and
$G$ in $e/\mathrm{Å}$ is in the same units, so $s\equiv1$ is a physical statement
rather than an arbitrary choice.

**Required input files** (same directory as this notebook): `6o2h.cif`,
`6o2h-sf.cif`, and `triclinic_lysozyme_maps.h5` (1.94 GB, from CXIDB).

In [ ]:
import io
import pathlib
from itertools import product

import numpy as np
import h5py
import gemmi
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.lines import Line2D
import scipy.constants as const
from scipy.spatial import cKDTree
from scipy.linalg import eigh
from scipy.ndimage import zoom as nd_zoom
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components as sc_connected_components
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation as Rot
import imageio
import imageio.v2 as iio2
from skimage.measure import marching_cubes
from IPython.display import Image as IPImage, display
import gc

np.set_printoptions(precision=4, suppress=True)

def fig_to_image(fig, dpi=72):
    """Rasterize a matplotlib figure to an RGB array, for building GIFs."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    buf.seek(0)
    return iio2.imread(buf)[:, :, :3]


In [ ]:
PDB_FILE = pathlib.Path("6o2h.cif")
SF_CIF   = pathlib.Path("6o2h-sf.cif")
H5_FILE  = pathlib.Path("triclinic_lysozyme_maps.h5")

# CXIDB 128 stores three Bragg-subtracted "processed" estimates of the diffuse
# component plus a raw total-scattering map and simulated comparison maps. The
# variational estimate is the one used for the lattice-dynamics fit in the
# original paper, so it is the default here.
MAP_GROUP = "maps/processed/variational"

D_MIN_MODEL    = 1.5    # Å — resolution for the visualization/reference density
RATE_MODEL     = 1.5
CONTACT_CUTOFF = 4.0    # Å — atom-atom cutoff defining a rigid-body contact
MIN_CONTACTS   = 5      # minimum atom-atom pairs for a contact to count
MAX_IMAGE      = 1      # search ±MAX_IMAGE unit cells (verified sufficient below)
TEMPERATURE    = 300.0  # K — used only for the frequency-unit conversion

# --- Resolution range for the fit -----------------------------------------
# Set deliberately, not by an SNR heuristic. Two reasons the heuristic used
# previously (mean(I)/mean(sigma) per shell, take the best contiguous run) was
# the wrong instrument:
#   * it is not an SNR. Diffuse intensity rises steeply with |q| (~q²|F|²), so
#     the ratio climbs with resolution mostly because the NUMERATOR grows. It
#     therefore always selects the finest shell available.
#   * high resolution is where this model is least valid. At ~2 Å the diffuse
#     scattering from a protein crystal is dominated by internal and side-chain
#     motion, not rigid-body lattice dynamics -- a central finding of the paper
#     this data comes from -- and it is also where the one-phonon approximation
#     degrades (multi-phonon terms grow as (q·u)^4).
# D_FIT_MIN is thus a model-validity statement. The SNR scan is retained below
# as a diagnostic, and R/CC are reported per resolution shell so the resolution
# at which the model stops working is measured rather than assumed.
D_FIT_MIN, D_FIT_MAX = 3.0, 8.0

N_FIT_MAX     = 6000    # mid-zone points used in the refinement
HOLDOUT_FRAC  = 0.15    # fraction held out for R_free
N_RES_STRATA  = 6       # equal-count resolution strata for the mid-zone sample
N_HALO_STRATA = 8       # equal-count Bragg-distance strata for the mid-zone sample

# A voxel counts as Bragg-adjacent only when it is close to a reciprocal-lattice
# point in ALL THREE indices. Expressed as a Cartesian radius, which is the
# physical criterion; an index-unit threshold interacts badly with the map's
# (1/11, 1/11, 1/13) sampling.
BRAGG_EXCL_CART = 0.010     # Å⁻¹

# --- Halo-profile term -----------------------------------------------------
# Rather than down-weighting individual voxels by distance from a Bragg peak,
# the halo information enters as an azimuthally-and-lattice-averaged PROFILE.
# Averaging beats the noise down by ~sqrt(N), turning ~10^6 noisy voxels (a
# fifth of them negative) into a few dozen well-determined numbers, and the
# resulting I ~ |q-G|^-2 shape with its anisotropic prefactor reads out the
# long-wavelength limit of D(q) directly -- exactly the part this data can
# determine. It also sidesteps the negative-intensity problem entirely, since
# shell averages are positive wherever there is real signal.
HALO_R_MAX    = 0.060   # Å⁻¹ — outer radius of the halo region
N_HALO_SHELLS = 14      # radial shells
N_HALO_DIRS   = 6       # angular sectors, so the profile keeps its anisotropy
HALO_WEIGHT   = 1.0     # relative weight of the halo term vs. the mid-zone term

# --- Prior and regularization ---------------------------------------------
# 1 k_BT/Å² = 0.414 N/m per atom pair, a reasonable order of magnitude for a weak
# non-covalent contact (vdW ≈ 1 N/m, H-bond ≈ 10-100 N/m). Only the prior's
# SHAPE (contact-count scaling, κ_R/κ_T ratio, patch anisotropy) is load-bearing;
# the overall magnitude is refined away by the shape normalization below.
PRIOR_K_PER_PAIR = 1.0
LAMBDA_PRIOR     = 1e-2   # weight on ||K - K_prior||²; set from the Gauss-Newton
                          # spectrum (see the sloppiness section)

CHUNK_EVAL = 20000      # q-points per batch in the vectorized evaluators
BZ_NGRID   = 20         # Brillouin-zone grid for the ADP integral (convergence
                        # is checked explicitly, never assumed)
SIGMA_FLOOR_PCT = 5.0   # percentile floor on sigma, so a few anomalously small
                        # error bars cannot dominate the whole objective

np.set_printoptions(precision=4, suppress=True)

In [ ]:
st = gemmi.read_structure(str(PDB_FILE))
st.remove_hydrogens()
st.remove_waters()
st.setup_entities()

cell = st.cell
a1 = np.array(cell.orthogonalize(gemmi.Fractional(1,0,0)).tolist())
a2 = np.array(cell.orthogonalize(gemmi.Fractional(0,1,0)).tolist())
a3 = np.array(cell.orthogonalize(gemmi.Fractional(0,0,1)).tolist())
A_orth  = np.column_stack([a1, a2, a3])       # orthogonalization matrix
B_recip = 2*np.pi * np.linalg.inv(A_orth).T   # reciprocal lattice (columns)

print(f"Cell: {cell.a:.3f}×{cell.b:.3f}×{cell.c:.3f} Å  "
      f"α={cell.alpha:.2f} β={cell.beta:.2f} γ={cell.gamma:.2f}°")
print(f"Space group: {st.spacegroup_hm}   Volume: {cell.volume:.1f} Å³")


## Experimental Map Metadata

The deposited stiffness model must be evaluated on **exactly** the same
$(h,k,l)$ grid the experimental map is sampled on for the two to be
compared voxel-by-voxel rather than merely visually. CXIDB 128 stores each
map group's grid as `grid_ori` (fractional Miller index of the first array
element) and `grid_delta` (spacing between array elements along each axis) —
i.e. the reciprocal-lattice vectors are subdivided by an independent integer
factor $(P_1,P_2,P_3)$ along $\mathbf a^*,\mathbf b^*,\mathbf c^*$
($13\times11\times11$ in the original deposition). Everything below reads
$(P_1,P_2,P_3)$ directly from the file's own metadata — rather than assuming
a value — and uses it consistently for the model's own fine reciprocal grid,
so that a model-evaluated point and an experimental voxel at the same
$(h,k,l)$ are the same physical point, not merely nearby ones.


In [ ]:
def _h5_floats(x):
    """Robustly pull one or more Python floats out of an HDF5 attribute,
    regardless of whether h5py hands it back as a bare Python/NumPy scalar,
    a 0-d array, or a length-N array (MATLAB-written attributes, as here,
    are commonly stored as arrays even for a single value)."""
    return [float(v) for v in np.atleast_1d(np.asarray(x)).ravel()]

def _h5_float(x):
    return _h5_floats(x)[0]

def _h5_str(x):
    """Robustly pull a Python str out of an HDF5 attribute that may come
    back as bytes, a NumPy bytes_/str_ scalar, or a length-1 array of any
    of those."""
    v = np.atleast_1d(np.asarray(x)).ravel()[0]
    return v.decode() if isinstance(v, bytes) else str(v)

with h5py.File(H5_FILE, 'r') as f:
    xattrs = dict(f['/crystal'].attrs)
    grp = f[f'/{MAP_GROUP}']
    grid_size  = tuple(int(round(v)) for v in _h5_floats(grp.attrs['grid_size']))
    grid_ori   = tuple(_h5_floats(grp.attrs['grid_ori']))
    grid_delta = tuple(_h5_floats(grp.attrs['grid_delta']))
    map_description = _h5_str(grp.attrs['Description']) if 'Description' in grp.attrs else MAP_GROUP
    map_units = _h5_str(grp.attrs['units']) if 'units' in grp.attrs else 'unknown'

    # The grid_size/grid_ori/grid_delta attributes describe the dataset's
    # axes in a fixed order, but the dataset's own on-disk axis order isn't
    # guaranteed to match: HDF5 files written from MATLAB (column-major)
    # commonly come out axis-reversed once read by h5py (row-major), so the
    # array's actual .shape can be grid_size reversed rather than grid_size
    # itself. Reconcile against the real dataset shape rather than trusting
    # the attribute order blindly.
    actual_shape = tuple(grp['I'].shape)
    if actual_shape == grid_size:
        pass
    elif actual_shape == grid_size[::-1]:
        print(f"NOTE: dataset 'I' shape {actual_shape} is axis-reversed relative to "
              f"the grid_size/grid_ori/grid_delta attribute order {grid_size} -- "
              f"reversing the metadata to match the file's actual on-disk axis order.")
        grid_size, grid_ori, grid_delta = grid_size[::-1], grid_ori[::-1], grid_delta[::-1]
    else:
        raise ValueError(f"Dataset 'I' shape {actual_shape} matches neither the "
                          f"grid_size attribute {grid_size} nor its reverse -- "
                          f"inspect '/{MAP_GROUP}' in the file manually.")
    assert tuple(grp['sigma'].shape) == actual_shape, \
        "'I' and 'sigma' datasets have different shapes -- unexpected file layout"

cell_h5 = dict(a=_h5_float(xattrs['a']), b=_h5_float(xattrs['b']), c=_h5_float(xattrs['c']),
               alpha=_h5_float(xattrs['alpha']), beta=_h5_float(xattrs['beta']),
               gamma=_h5_float(xattrs['gamma']), sg=int(round(_h5_float(xattrs['spaceGroupNumber']))))

print(f"h5 crystal: a={cell_h5['a']:.3f} b={cell_h5['b']:.3f} c={cell_h5['c']:.3f}  "
      f"α={cell_h5['alpha']:.2f} β={cell_h5['beta']:.2f} γ={cell_h5['gamma']:.2f}  "
      f"SG#{cell_h5['sg']}")
print(f"Map group '/{MAP_GROUP}': {map_description}  [{map_units}]")
print(f"grid_size={grid_size}  grid_ori={grid_ori}  grid_delta={grid_delta}")

for name, dep, exp in [('a',cell.a,cell_h5['a']), ('b',cell.b,cell_h5['b']), ('c',cell.c,cell_h5['c']),
                        ('alpha',cell.alpha,cell_h5['alpha']), ('beta',cell.beta,cell_h5['beta']),
                        ('gamma',cell.gamma,cell_h5['gamma'])]:
    rel_err = abs(dep-exp)/max(abs(dep), 1e-6)
    flag = '  <-- MISMATCH' if rel_err > 5e-3 else ''
    print(f"  {name}: deposited {dep:.4f}  vs  h5 {exp:.4f}{flag}")

# P1,P2,P3: integer subdivision of each reciprocal axis, read from the data
# itself rather than assumed.
P1 = int(round(1.0/grid_delta[0]))
P2 = int(round(1.0/grid_delta[1]))
P3 = int(round(1.0/grid_delta[2]))
assert abs(1.0/P1 - grid_delta[0]) < 1e-4, "grid_delta[0] is not a clean 1/integer subdivision"
assert abs(1.0/P2 - grid_delta[1]) < 1e-4, "grid_delta[1] is not a clean 1/integer subdivision"
assert abs(1.0/P3 - grid_delta[2]) < 1e-4, "grid_delta[2] is not a clean 1/integer subdivision"
print(f"q-resolution matched to the experimental map: (P1,P2,P3) = ({P1},{P2},{P3})")

def hkl_index(h, k, l):
    """Nearest (i, j, k_idx) integer array index for fractional Miller
    index (h,k,l) on this map group's grid; also returns how far off the
    nearest grid point actually was (should be ~0 for points our own model
    grid produces)."""
    fi = (h - grid_ori[0]) / grid_delta[0]
    fj = (k - grid_ori[1]) / grid_delta[1]
    fk = (l - grid_ori[2]) / grid_delta[2]
    i, j, kx = round(fi), round(fj), round(fk)
    err = max(abs(fi-i), abs(fj-j), abs(fk-kx))
    return i, j, kx, err

def in_bounds(i, j, k):
    return (0 <= i < grid_size[0]) and (0 <= j < grid_size[1]) and (0 <= k < grid_size[2])


## Atomic Model, Mass Matrix, and the Molecular Transform

Everything downstream is referenced to **one** point: `r_cm_at`, the atomic
centre of mass. That choice is forced, not cosmetic — the mass matrix
$M=\mathrm{diag}(J,\,mI)$ is block-diagonal (no translation–rotation coupling)
only about the centre of mass, and a rigid displacement
$\delta\mathbf r=\mathbf v+\boldsymbol\Omega\times(\mathbf r-\mathbf r_{\rm ref})$
splits into $(\boldsymbol\Omega,\mathbf v)$ differently for every choice of
$\mathbf r_{\rm ref}$. If $G$ and $D$ are built about different reference
points, $I=G^\dagger D^{-1}G$ contracts two incompatible coordinate systems.

**Why the molecular transform is computed atom-by-atom, not from a density
grid.** The crystal density on a periodic grid is
$\rho_{\rm cell}(\mathbf r)=\sum_{\mathbf n}\rho_{\rm mol}(\mathbf r-\mathbf R_{\mathbf n})$
folded into one box, so

$$F_{\rm cell}(\mathbf q)=\sum_{\mathbf n}e^{-i\mathbf q\cdot\mathbf R_{\mathbf n}}
\int_{\rm box-\mathbf R_{\mathbf n}}\rho_{\rm mol}(\mathbf r')e^{-i\mathbf q\cdot\mathbf r'}d^3r'.$$

At **integer** $hkl$ every phase factor is 1 and $F_{\rm cell}=F_{\rm mol}$. At
the **fractional** $hkl$ this whole pipeline runs on, they are not, so
$F_{\rm cell}\neq F_{\rm mol}$ for any molecule that crosses a cell boundary —
and for a 14 kDa protein in a 27x32x34 Å cell there is no roll of the grid that
leaves empty margins, because crystal contacts are contiguous by construction.
Summing over the deposited (unwrapped) coordinates with IT92 form factors is
exact, unambiguous about $\mathbf r_{\rm cm}$, and roughly 300x faster
(1001 atoms vs. 311040 voxels per $\mathbf q$-point).

The experimentally measured Bragg amplitudes are still used, but as a smooth
**resolution-dependent amplitude correction** $\langle|F_{\rm meas}|\rangle/
\langle|F_{\rm calc}|\rangle$ applied to the atomic transform, rather than by
building a hybrid density that would reintroduce the wrapping problem.


In [ ]:
# --- Atomic model: positions, masses, deposited ADPs, mass matrix ----------
# Built FIRST, because r_cm_at is the reference point for G, for D, and for the
# ADP projection alike (see markdown above).
ATOMIC_MASS = {'C':12.011,'N':14.007,'O':15.999,'S':32.06,'P':30.974,
               'SE':78.96,'H':1.008,'FE':55.845,'ZN':65.38,'CA':40.078}

masses, apos, b_exp, u_exp, has_aniso, elements = [], [], [], [], [], []
for ch in st[0]:
    for res in ch:
        for atom in res:
            masses.append(ATOMIC_MASS.get(atom.element.name.upper(), 12.0))
            apos.append(atom.pos.tolist())
            b_exp.append(atom.b_iso)
            u_exp.append(atom.aniso.as_mat33().tolist())
            has_aniso.append(atom.aniso.nonzero())
            elements.append(atom.element)
masses = np.array(masses); apos = np.array(apos); b_exp = np.array(b_exp)
u_exp = np.array(u_exp); has_aniso = np.array(has_aniso)
all_pos = apos                      # same array, used by the contact search below
n_atoms = len(apos)

m_total = masses.sum()
r_cm_at = (masses[:, None]*apos).sum(0)/m_total     # THE reference point
dr_a    = apos - r_cm_at
r2      = (dr_a**2).sum(1)
J       = (masses[:, None, None]*(r2[:, None, None]*np.eye(3)[None]
          - dr_a[:, :, None]*dr_a[:, None, :])).sum(0)
M_mat   = np.block([[J, np.zeros((3,3))], [np.zeros((3,3)), m_total*np.eye(3)]])
Msq     = np.linalg.cholesky(M_mat)
Msq_inv = np.linalg.inv(Msq)

kBT_SI     = const.k * TEMPERATURE
omega_unit = np.sqrt(kBT_SI / (const.atomic_mass * (1e-10)**2))
freq_unit  = omega_unit / (2*np.pi) / 1e12   # THz per sqrt(reduced stiffness/mass)

print(f"{n_atoms} atoms   total mass {m_total:.0f} amu")
print(f"Centre of mass r_cm_at = ({r_cm_at[0]:.3f}, {r_cm_at[1]:.3f}, {r_cm_at[2]:.3f}) Å")
print(f"Deposited ANISOU records present for {has_aniso.mean():.1%} of atoms")
print(f"Deposited mean B = {b_exp.mean():.2f} Å²")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")

# The molecule must not be split across the cell boundary in the deposited
# coordinates either -- check its extent against the cell, since a wrapped
# deposition would break the atomic transform in exactly the same way a
# periodic density grid does.
frac = np.linalg.solve(A_orth, (apos - r_cm_at).T).T
span = frac.max(0) - frac.min(0)
print(f"Molecular extent in fractional coordinates: {span.round(3)} "
      f"({'OK, contiguous' if (span < 1.0).all() else 'WARNING: spans a full cell axis'})")

# --- IT92 form-factor coefficients, one row per atom -----------------------
def _it92_coeffs(el):
    """(a, b, c) for one element, tolerant of gemmi's 4- vs 5-Gaussian layouts."""
    t = el.it92
    a = np.asarray(list(t.a), float)
    b = np.asarray(list(t.b), float)
    c = float(getattr(t, 'c', 0.0))
    return a, b, c

_ab = [_it92_coeffs(el) for el in elements]
_na = max(len(a) for a, b, c in _ab)
FF_A = np.zeros((n_atoms, _na)); FF_B = np.zeros((n_atoms, _na)); FF_C = np.zeros(n_atoms)
for i, (a, b, c) in enumerate(_ab):
    FF_A[i, :len(a)] = a; FF_B[i, :len(b)] = b; FF_C[i] = c
Z_atomic = (FF_A.sum(1) + FF_C).sum()
print(f"Σ f(0) over all atoms = {Z_atomic:.1f} e⁻  (electron count of the model)")


In [ ]:
# --- Model density: used ONLY for the molecular isosurface at the end of the
# notebook, and to derive the measured/calculated amplitude correction below.
# It is deliberately NOT used to compute G(q): see the markdown above for why a
# periodic density grid gives the wrong F(q) at non-integer (h,k,l).
dc = gemmi.DensityCalculatorX()
dc.d_min = D_MIN_MODEL; dc.rate = RATE_MODEL
dc.set_grid_cell_and_spacegroup(st)
dc.initialize_grid()
dc.add_model_density_to_grid(st[0])
dc.grid.symmetrize_sum()
rho_model = np.array(dc.grid)
print(f"Visualization density grid: {rho_model.shape[0]}×{rho_model.shape[1]}×{rho_model.shape[2]}")


In [ ]:
# --- Molecular transform F(q), L(q) by direct atomic summation -------------
#
#   F(q) = Σ_a f_a(|q|) e^{-i q·r_a}            (electrons)
#   L(q) = Σ_a f_a(|q|) (r_a - r_cm_at) e^{-i q·r_a}   (electrons · Å)
#
# USE_ADP_IN_TRANSFORM:
#   True  (default) -- multiply each atom by exp(-B_a s²), giving the
#         Debye-Waller-smeared transform, i.e. the same object the measured
#         Bragg amplitudes represent. This is the conventional choice and keeps
#         the amplitude correction below self-consistent.
#         CAVEAT: the deposited B already contains the lattice motion that
#         D(q)^-1 is also modelling, so this mildly double-counts Debye-Waller
#         at the one-phonon level. The effect is small at the resolutions where
#         this model is valid; set to False to see the size of it.
#   False -- the rigid, unsmeared molecular transform.
USE_ADP_IN_TRANSFORM = True

def _form_factors(s2):
    """f_a(s²) for every atom, s = |q|/4π. Returns (n_q, n_atoms)."""
    f = (FF_A[None, :, :] * np.exp(-FF_B[None, :, :] * s2[:, None, None])).sum(-1) \
        + FF_C[None, :]
    if USE_ADP_IN_TRANSFORM:
        f = f * np.exp(-b_exp[None, :] * s2[:, None])
    return f

# Amplitude correction: replace the *magnitude* of the calculated transform
# with the measured one, as a smooth function of resolution. This is how the
# experimental Bragg amplitudes enter, without building a hybrid density (which
# would reintroduce the unit-cell wrapping error, see markdown above).
APPLY_AMPLITUDE_CORRECTION = True
N_AMP_BINS = 24

def _build_amplitude_correction():
    doc_sf = gemmi.cif.read(str(SF_CIF))
    rb     = gemmi.as_refln_blocks(doc_sf)[0]
    mil    = np.array(rb.make_miller_array())
    Fm     = np.array(rb.make_float_array("F_meas_au"))
    ok     = np.isfinite(Fm) & (Fm > 0)
    mil, Fm = mil[ok], Fm[ok]
    q  = mil @ B_recip.T
    qn = np.linalg.norm(q, axis=1)
    s2 = (qn/(4*np.pi))**2
    f  = _form_factors(s2)
    Fc = np.abs(np.einsum('na,na->n', f, np.exp(-1j*(q @ apos.T))))
    edges = np.quantile(qn, np.linspace(0, 1, N_AMP_BINS+1))
    edges[0] = 0.0; edges[-1] = qn.max()*1.5
    idx = np.clip(np.digitize(qn, edges)-1, 0, N_AMP_BINS-1)
    mid, rat = [], []
    for b in range(N_AMP_BINS):
        m = idx == b
        if m.sum() < 20:
            continue
        mid.append(0.5*(edges[b]+edges[b+1]))
        rat.append(Fm[m].mean()/max(Fc[m].mean(), 1e-12))
    mid, rat = np.array(mid), np.array(rat)
    print(f"Amplitude correction from {len(Fm)} reflections, {len(mid)} usable shells; "
          f"|F_meas|/|F_calc| ranges {rat.min():.3f}–{rat.max():.3f}")
    return mid, rat

if APPLY_AMPLITUDE_CORRECTION:
    _AMP_Q, _AMP_R = _build_amplitude_correction()
    def amplitude_correction(qn):
        return np.interp(qn, _AMP_Q, _AMP_R, left=_AMP_R[0], right=_AMP_R[-1])
else:
    def amplitude_correction(qn):
        return np.ones_like(qn)

def F_L_batch(h_arr, k_arr, l_arr, chunk=4000):
    """F(q) and L(q) at arbitrary fractional Miller indices. Memory is bounded
    by an (chunk × n_atoms) phase matrix -- tiny, since n_atoms ≈ 10³."""
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    N = len(h_arr)
    F_out = np.zeros(N, complex); L_out = np.zeros((N, 3), complex)
    d = apos - r_cm_at
    for s in range(0, N, chunk):
        sl = slice(s, min(s+chunk, N))
        q  = (h_arr[sl, None]*B_recip[:, 0] + k_arr[sl, None]*B_recip[:, 1]
              + l_arr[sl, None]*B_recip[:, 2])
        qn = np.linalg.norm(q, axis=1)
        w  = _form_factors((qn/(4*np.pi))**2) * np.exp(-1j*(q @ apos.T))
        w *= amplitude_correction(qn)[:, None]
        F_out[sl] = w.sum(1)
        L_out[sl] = w @ d
    return F_out, L_out

_F0, _L0 = F_L_batch([0.0], [0.0], [0.0])
print(f"F(000) = {_F0[0].real:.1f} e⁻   (Σ f(0) = {Z_atomic:.1f})")
print(f"L(000) = {_L0[0].real.round(2)} e⁻·Å   "
      f"(the electron first moment about the MASS centre; small, not exactly zero)")


## Coupling Vector $G(\mathbf{q})$

$$G = \begin{pmatrix}G_R\\G_T\end{pmatrix}
    = \begin{pmatrix}i\mathbf{q}\times L(\mathbf{q})\\
                      i\mathbf{q}\,F(\mathbf{q})\end{pmatrix},
\quad F = \sum_a f_a e^{-i\mathbf{q}\cdot\mathbf{r}_a},
\quad L = \sum_a f_a(\mathbf{r}_a-\mathbf{r}_{\rm cm})\,e^{-i\mathbf{q}\cdot\mathbf{r}_a}$$

with $\mathbf r_{\rm cm}=$ `r_cm_at`, the same point the dynamical matrix uses.

$G$ is **complex**, and it stays complex everywhere below. $I=G^\dagger D^{-1}G$
is a Hermitian quadratic form; replacing $G$ by $\mathrm{Re}\,G$ (or $D$ by
$\mathrm{Re}\,D$) is not an approximation but a different quantity — on this
geometry it costs a median factor of $\approx 0.46$ in $I(\mathbf q)$, with a
5x spread. Two consistency checks below guard against that class of error.


In [ ]:
def G_at_hkl_batch(h_arr, k_arr, l_arr):
    """Vectorized G(q) at arrays of fractional Miller indices. (N, 6) complex."""
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    q  = (h_arr[:, None]*B_recip[:, 0] + k_arr[:, None]*B_recip[:, 1]
          + l_arr[:, None]*B_recip[:, 2])
    iq = 1j*q
    F, L = F_L_batch(h_arr, k_arr, l_arr)
    return np.concatenate([np.cross(iq, L), iq*F[:, None]], axis=1)

def G_at_hkl(h, k, l):
    return G_at_hkl_batch([h], [k], [l])[0]

_Gt = G_at_hkl(1.5, -0.75, 0.25)
print("G at a representative fractional hkl (complex, 6 components):")
print("  |Re G| =", np.abs(_Gt.real).round(1))
print("  |Im G| =", np.abs(_Gt.imag).round(1))
assert np.abs(_Gt.imag).max() > 1e-6*np.abs(_Gt).max(), \
    "G has no imaginary part -- something has silently realified the transform"
print("OK: G retains a substantial imaginary part, as it must.")


## Rigid-Body Contacts

Two unit-cell images are in contact if any of their atoms come within
`CONTACT_CUTOFF`. Born–von Kármán translational symmetry means the contact
between cell 0 and cell $\mathbf n$ depends only on $\mathbf n$, and the two
directions of a contact ($\mathbf n$ and $-\mathbf n$) are the same physical
spring seen from either side — which is why 12 directed images collapse to 6
independent stiffnesses.

Note this is *not* a point-group symmetry reduction: in $P1$ there is no point
symmetry at all, so the six contacts are genuinely **distinct**, not
"symmetry-distinct". They share nothing, and each carries its own free
$6\times6$ matrix.

For each contact we record the centroid of the atom pairs actually within the
cutoff — the physical location of the contact patch — which sets both the
contact-local reference frame and the geometric prior on $K$.


In [ ]:
tree = cKDTree(all_pos)

def canonical(n):
    return min(n, tuple(-x for x in n))

def _find_interfaces(max_image):
    found = {}
    for n_tup in product(range(-max_image, max_image+1), repeat=3):
        if n_tup == (0,0,0):
            continue
        R_n = sum(n_tup[i]*[a1,a2,a3][i] for i in range(3))
        idx_lists = tree.query_ball_point(all_pos + R_n, CONTACT_CUTOFF)
        n_c = sum(len(idx) for idx in idx_lists)
        if n_c >= MIN_CONTACTS:
            found[n_tup] = {'R_n': R_n, 'n_contacts': n_c, 'idx_lists': idx_lists}
    return found

raw_interfaces = _find_interfaces(MAX_IMAGE)

# Verify MAX_IMAGE is actually large enough rather than assuming it. One of the
# detected contacts here sits at |R_n| ≈ 47 Å, so the molecule is long enough
# that second-shell images are worth ruling out explicitly.
_outer = {n: v for n, v in _find_interfaces(MAX_IMAGE+1).items()
          if max(abs(x) for x in n) > MAX_IMAGE}
if _outer:
    print(f"WARNING: {len(_outer)} contact(s) found beyond MAX_IMAGE={MAX_IMAGE}: "
          f"{sorted(_outer)} -- increase MAX_IMAGE.")
else:
    print(f"Checked shell {MAX_IMAGE+1}: no additional contacts, MAX_IMAGE={MAX_IMAGE} is sufficient.")

unique = {}
for n_tup, info in raw_interfaces.items():
    c = canonical(n_tup)
    if c not in unique:
        unique[c] = dict(info, n_tup=n_tup)

def contact_midpoints(R_n, idx_lists):
    """Midpoints (lab frame) of every atom pair within CONTACT_CUTOFF."""
    j_idx = np.concatenate([np.full(len(ii), j, int) for j, ii in enumerate(idx_lists) if len(ii)]) \
            if any(len(ii) for ii in idx_lists) else np.zeros(0, int)
    i_idx = np.concatenate([np.asarray(ii, int) for ii in idx_lists if len(ii)]) \
            if any(len(ii) for ii in idx_lists) else np.zeros(0, int)
    return 0.5*(all_pos[i_idx] + all_pos[j_idx] + R_n)

for c, info in unique.items():
    info['midpoints']  = contact_midpoints(info['R_n'], info['idx_lists'])
    info['contact_pt'] = info['midpoints'].mean(0)

shell_order = sorted(unique.keys())
print(f"\nDetected {len(unique)} distinct contacts ({len(raw_interfaces)} directed images):")
for c in shell_order:
    info = unique[c]
    d = info['midpoints'] - info['contact_pt']
    rg = np.sqrt((d**2).sum(1).mean())
    print(f"  n={c}  |R_n|={np.linalg.norm(info['R_n']):6.2f} Å  "
          f"atom-pairs={info['n_contacts']:3d}  patch gyration radius={rg:5.2f} Å")


## Reduced Units

Stiffnesses are expressed directly in units of $k_BT$, so $k_BT=1$ throughout.

One consequence is worth stating because it is a free correctness test: in the
classical limit $\langle u_{\mathbf q}u_{\mathbf q}^\dagger\rangle=k_BT\,
D(\mathbf q)^{-1}$, which contains **no mass matrix at all**. $M$ therefore
affects only the plotted frequencies — never $I(\mathbf q)$, never the ADPs. At
room temperature the classical limit is amply justified: the branches sit below
$\sim0.4$ THz, i.e. $\hbar\omega\lesssim1.6$ meV against $k_BT=25.9$ meV.

This also fixes the absolute units of $I_{\rm model}$. With $k_BT=1$ and $G$ in
$e/\mathrm{Å}$, $G^\dagger D^{-1}G$ is in electrons² per unit cell — the same
scale the deposited map reports ("electron scattering per unit cell"). That
coincidence is what lets the $K$/scale degeneracy be closed without touching the
ADPs; see the refinement section.

In [ ]:
# The mass matrix was built alongside the atomic model above (it has to be, since
# r_cm_at is also G's reference point). Report it here and check the classical
# limit explicitly.
print(f"Moment of inertia J (amu·Å²), eigenvalues: {np.linalg.eigvalsh(J).round(0)}")
print(f"Total mass: {m_total:.0f} amu")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")

hbar_over_kT_THz = const.hbar*2*np.pi*1e12/(const.k*TEMPERATURE)
print(f"\nClassical-limit check: ħω/k_BT = {hbar_over_kT_THz:.4f} × (ω in THz)")
print(f"  at 0.4 THz -> {0.4*hbar_over_kT_THz:.3f};  at 2 THz -> {2*hbar_over_kT_THz:.3f}")
print("  Both « 1, so the classical (equipartition) form is appropriate.")

## Contact Frames, the Geometric Prior, and the Stiffness Parameterization

Each contact's $6\times6$ stiffness $K$ is specified in a **local contact
frame**: origin at the measured contact centroid, $z$-axis along
$\mathbf R_{\mathbf n}$. A fixed rigid shift-and-rotation (computed once from
the structure, never fit) re-expresses $K$ at the far body's own centre of mass,
which is the representation the dynamical matrix uses.

**The prior.** Model the contact as $n$ independent isotropic point springs of
stiffness $k$ sitting at the atom-pair midpoints $\mathbf m_i$, with
$\mathbf d_i=\mathbf m_i-\bar{\mathbf m}$:

$$E=\tfrac{k}{2}\sum_i\bigl|\mathbf v+\boldsymbol\Omega\times\mathbf d_i\bigr|^2
\;\Longrightarrow\;
K_{TT}=k\,n\,I_3,\qquad
K_{RR}=k\sum_i\bigl(|\mathbf d_i|^2I-\mathbf d_i\mathbf d_i^{\mathsf T}\bigr),\qquad
K_{TR}=0$$

the last exactly, because $\sum_i\mathbf d_i=0$ at the centroid. This costs no
extra parameters and supplies four things a flat isotropic guess cannot:

* **scaling with the number of atom–atom contacts** (9 to 63 here, a 7x spread);
* **the correct $\kappa_R/\kappa_T$ ratio**, which is $\sim\rho_g^2$ with
  $\rho_g$ the patch gyration radius. Setting $\kappa_R=\kappa_T=1$ — one in
  $k_BT/\mathrm{rad}^2$, the other in $k_BT/\mathrm{Å}^2$ — makes the
  librational springs roughly $25$–$60\times$ too soft, and the librational
  branches are exactly the ones that go pathological in an unconstrained fit;
* **the correct patch anisotropy** (a flat contact is automatically soft about
  axes lying in its own plane);
* **a vanishing local-frame $T$–$R$ block**, which is the property an ad-hoc
  regularizer was previously trying to impose by hand.

The refinement is then regularized **toward this prior**, not toward zero. For
a sloppy model that distinction is decisive: penalizing only the $T$–$R$ block
leaves $\sim18$ directions per contact — including every overall magnitude —
with nothing holding them, so they drift to whichever boundary the optimizer
reaches first.

**Stability.** $K=LL^{\mathsf T}$ with $L$ lower-triangular and otherwise free
is positive semidefinite for any real $L$, so the optimizer runs unconstrained.
And that is *sufficient* for the whole crystal, because each contact enters the
dynamical matrix only as

$$D_{\mathbf n}(\mathbf q)=M_{\mathbf n}(\mathbf q)^\dagger K_{\mathbf n}
M_{\mathbf n}(\mathbf q),\qquad
M_{\mathbf n}(\mathbf q)=A_{\mathbf n}-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}I$$

which is manifestly Hermitian PSD. (An earlier version of this note claimed the
contribution collapses to $2K(1-\cos\mathbf q\cdot\mathbf R_{\mathbf n})$; it
does not — expanding leaves $B^{\mathsf T}KB$ and cross terms with
$B=A-I$. The $M^\dagger KM$ form above is the correct identity, and it makes
the conclusion immediate rather than approximate.)


In [ ]:
def skew(v):
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])

def contact_frame(R_n):
    """Orthonormal frame with z along R_n."""
    ez = R_n / np.linalg.norm(R_n)
    perp = np.array([1.,0.,0.]) if abs(ez[0]) < 0.9 else np.array([0.,1.,0.])
    ex = np.cross(perp, ez); ex /= np.linalg.norm(ex)
    return np.column_stack([ex, np.cross(ez, ex), ez])

def Gc_inv_matrix(c):
    """Inverse of the rigid-shift adjoint that moves a reference point by c."""
    G = np.eye(6); G[3:6, 0:3] = -skew(c)
    return G

def K_local_to_lab(K_local, R_frame, c_local):
    """K_local is expressed at the contact centroid, in local-frame axes.
    Returns K expressed at the far body's own centre of mass, in lab axes."""
    Gc_inv = Gc_inv_matrix(c_local)
    K_shifted = Gc_inv.T @ K_local @ Gc_inv
    Gamma = np.block([[R_frame, np.zeros((3,3))],[np.zeros((3,3)), R_frame]])
    return Gamma @ K_shifted @ Gamma.T

for c in shell_order:
    info = unique[c]
    R_frame = contact_frame(info['R_n'])
    d_c     = info['contact_pt'] - r_cm_at        # contact offset from body-0 COM
    info['R_frame'] = R_frame
    info['c_local'] = R_frame.T @ (info['R_n'] - d_c)
    info['Gamma']   = np.block([[R_frame, np.zeros((3,3))],
                                [np.zeros((3,3)), R_frame]])

TRIL_I, TRIL_J = np.tril_indices(6)
N_TRIL   = len(TRIL_I)                # 21
N_SHELL  = len(shell_order)
N_PARAMS = N_SHELL * N_TRIL

def theta_to_Lmap(theta):
    Lmap, idx = {}, 0
    for c in shell_order:
        L = np.zeros((6,6))
        L[TRIL_I, TRIL_J] = theta[idx:idx+N_TRIL]
        Lmap[c] = L
        idx += N_TRIL
    return Lmap

def Lmap_to_theta(Lmap):
    theta, idx = np.zeros(N_PARAMS), 0
    for c in shell_order:
        theta[idx:idx+N_TRIL] = Lmap[c][TRIL_I, TRIL_J]
        idx += N_TRIL
    return theta

def build_K(Lmap):
    """Cholesky factors -> (K in the local contact frame, K in the lab frame at
    each contact's far-body COM, the representation dynamical_matrix expects)."""
    K_local, K_lab = {}, {}
    for c in shell_order:
        Kl = Lmap[c] @ Lmap[c].T
        K_local[c] = Kl
        K_lab[c] = K_local_to_lab(Kl, unique[c]['R_frame'], unique[c]['c_local'])
    return K_local, K_lab

# --- Geometric prior: n isotropic point springs at the atom-pair midpoints ---
def contact_patch_stiffness(info, k_per_pair):
    d = info['midpoints'] - info['contact_pt']
    n = len(d)
    K = np.zeros((6,6))
    K[3:6, 3:6] = k_per_pair * n * np.eye(3)                        # translation
    K[0:3, 0:3] = k_per_pair * ((d**2).sum()*np.eye(3) - d.T @ d)   # rotation
    return K                                                        # T-R block = 0

def prior_Lmap(k_per_pair=PRIOR_K_PER_PAIR, ridge_frac=1e-3):
    Lmap, K_prior_local = {}, {}
    for c in shell_order:
        info = unique[c]
        Gamma = info['Gamma']
        Kl = Gamma.T @ contact_patch_stiffness(info, k_per_pair) @ Gamma
        Kl = 0.5*(Kl + Kl.T)
        Kl += ridge_frac * np.trace(Kl)/6 * np.eye(6)   # keep strictly PD
        K_prior_local[c] = Kl
        Lmap[c] = np.linalg.cholesky(Kl)
    return Lmap, K_prior_local

L_PRIOR, K_PRIOR_LOCAL = prior_Lmap()
_, K_LAB_PRIOR = build_K(L_PRIOR)

def regularizer_prior(K_local, lam=LAMBDA_PRIOR):
    """lam * Σ_n ||K_n - K_n^prior||_F² / ||K_n^prior||_F² -- scale-free per contact."""
    return lam*sum(np.sum((K_local[c]-K_PRIOR_LOCAL[c])**2)/np.sum(K_PRIOR_LOCAL[c]**2)
                   for c in shell_order)

def regularizer_prior_dK(K_local, lam=LAMBDA_PRIOR):
    return {c: 2*lam*(K_local[c]-K_PRIOR_LOCAL[c])/np.sum(K_PRIOR_LOCAL[c]**2)
            for c in shell_order}

print(f"Free parameters: {N_SHELL} contacts × {N_TRIL} Cholesky entries = {N_PARAMS}\n")
print(f"Geometric prior at k_per_pair = {PRIOR_K_PER_PAIR} k_BT/Å² "
      f"(= {PRIOR_K_PER_PAIR*0.414:.2f} N/m per atom pair):")
for c in shell_order:
    Kl = K_PRIOR_LOCAL[c]
    kt, kr = np.trace(Kl[3:6,3:6])/3, np.trace(Kl[0:3,0:3])/3
    print(f"  {str(c):14s} κ_T={kt:8.2f} k_BT/Å²   κ_R={kr:9.1f} k_BT/rad²   "
          f"ratio={kr/kt:6.1f} Å²   |T-R|={np.abs(Kl[0:3,3:6]).max():.2e}")
print("\n(The ratio column is the point: a flat isotropic guess sets it to 1.)")


## Dynamical Matrix

For contact $\mathbf n$ with stiffness $K_{\mathbf n}$ (expressed at the far
body's own centre of mass) and adjoint
$A_{\mathbf n}=\left(\begin{smallmatrix}I&0\\-[\mathbf R_{\mathbf n}]_\times&I
\end{smallmatrix}\right)$,

$$D_{\mathbf n}(\mathbf q)=M_{\mathbf n}(\mathbf q)^\dagger K_{\mathbf n}
M_{\mathbf n}(\mathbf q),\qquad
M_{\mathbf n}(\mathbf q)=A_{\mathbf n}-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}I$$

and $D=\sum_{\mathbf n}D_{\mathbf n}$, summed over the 6 independent contacts
(both directions of each are already included). Building $D$ this way rather
than expanding into four terms makes it **exactly** Hermitian PSD in floating
point, so no cancellation error can push an acoustic eigenvalue negative.

$D(\mathbf q)$ is complex Hermitian, not real symmetric:
$\mathrm{Im}\,D_{\mathbf n}=\sin(\mathbf q\cdot\mathbf R_{\mathbf n})
\bigl(K_{\mathbf n}A_{\mathbf n}-(K_{\mathbf n}A_{\mathbf n})^{\mathsf T}\bigr)$,
generically $O(1)$. Every eigendecomposition below uses the full complex matrix.

**Zero modes at $\Gamma$.** $D(\mathbf q)u=0$ iff $M_{\mathbf n}(\mathbf q)u=0$
for every $\mathbf n$ (given $K_{\mathbf n}\succ0$). At $\mathbf q=0$ that reads
$\mathbf R_{\mathbf n}\times\boldsymbol\Omega=0$ for all $\mathbf n$, which for
three independent $\mathbf R_{\mathbf n}$ forces $\boldsymbol\Omega=0$ with
$\mathbf v$ free. So **exactly 3 eigenvalues vanish at $\Gamma$** — the pure
translations — and the 3 librational eigenvalues sit at whatever rest frequency
the rotational stiffness sets. A Born–von Kármán lattice with fixed
$\mathbf R_{\mathbf n}$ is not rotationally invariant, so a global rotation of
the crystal is not a zero mode of this model. That is a real limitation of the
model, not an error, but it is worth naming.


In [ ]:
def ad_translation(R_n):
    A = np.eye(6); A[3:6, 0:3] = -skew(R_n); return A

_I6 = np.eye(6)

def dynamical_matrix_batch(q_batch, unique, K_lab):
    """D(q) = Σ_n M_n(q)^† K_n M_n(q) over a batch of Cartesian q. (N,6,6) complex."""
    q_batch = np.atleast_2d(np.asarray(q_batch, float))
    D = np.zeros((len(q_batch), 6, 6), dtype=complex)
    for c, info in unique.items():
        R_n = info['R_n']; K = K_lab[c]
        A_n = ad_translation(R_n)
        phase = np.exp(1j*(q_batch @ R_n))                        # (N,)
        M = A_n[None,:,:] - phase[:,None,None]*_I6[None,:,:]      # (N,6,6)
        D += np.einsum('nji,jk,nkl->nil', M.conj(), K, M)
    return D

def dynamical_matrix(q_cart, unique, K_lab):
    return dynamical_matrix_batch(np.asarray(q_cart, float)[None,:], unique, K_lab)[0]

def mass_weighted_eigs(D_batch):
    """Eigenvalues of the pencil (D, M), via the Cholesky congruence. Complex
    Hermitian throughout -- taking .real here would be a different matrix."""
    Dw = np.einsum('ij,njk,lk->nil', Msq_inv, D_batch, Msq_inv)
    return np.linalg.eigvalsh(Dw)

# --- Sanity check at Gamma -------------------------------------------------
D0  = dynamical_matrix(np.zeros(3), unique, K_LAB_PRIOR)
ev0 = mass_weighted_eigs(D0[None])[0]
print("D(q=0) mass-weighted eigenvalues:", ev0.round(8))
print("  -> exactly 3 acoustic zeros expected; the upper 3 are the librational")
print("     rest frequencies and are NOT required to vanish (see markdown).")
assert np.sum(np.abs(ev0) < 1e-8*max(abs(ev0).max(), 1e-30)) == 3, \
    "expected exactly 3 zero modes at Gamma"

# --- Hermiticity / PSD check at a generic q --------------------------------
_qc = B_recip @ np.array([0.31, -0.17, 0.43])
_Dc = dynamical_matrix(_qc, unique, K_LAB_PRIOR)
print(f"\nAt a generic q: ||Im D|| / ||Re D|| = "
      f"{np.linalg.norm(_Dc.imag)/np.linalg.norm(_Dc.real):.4f}  (must be > 0)")
print(f"  Hermitian: {np.allclose(_Dc, _Dc.conj().T)}   "
      f"min eigenvalue: {np.linalg.eigvalsh(_Dc).min():.4e} (must be >= 0)")
print(f"  eigenvalues of Re(D) alone would be: {np.linalg.eigvalsh(_Dc.real)[:3].round(4)}")
print(f"  eigenvalues of the true complex D:   {np.linalg.eigvalsh(_Dc)[:3].round(4)}")
print("  -> these differ; using D.real is not an approximation but a different model.")


## Phonon Band Structure

Eigenvalues of the mass-weighted dynamical matrix along a path through the
Brillouin zone, using the geometric prior as the starting model. Batched and
fully complex-Hermitian: this function is called once per frame of the
convergence movie later, so the per-q Python loop dominated runtime.

In [ ]:
HSP = {'Γ':np.array([0.,0.,0.]),'X':np.array([.5,0.,0.]),
       'Y':np.array([0.,.5,0.]),'Z':np.array([0.,0.,.5])}
path_labels = ['Γ','X','Γ','Y','Γ','Z','Γ']
N_seg = 60

q_frac, tick_idx = [], [0]
for seg in range(len(path_labels)-1):
    p0, p1 = HSP[path_labels[seg]], HSP[path_labels[seg+1]]
    last = (seg == len(path_labels)-2)
    for t in np.linspace(0, 1, N_seg, endpoint=last):
        q_frac.append(p0*(1-t)+p1*t)
    if not last: tick_idx.append(len(q_frac))
tick_idx.append(len(q_frac)-1)

q_frac = np.array(q_frac)
q_cart = (B_recip @ q_frac.T).T
x = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(q_cart,axis=0),axis=1))])

def bandstructure_freqs(K_lab, qs=None):
    """Batched and fully complex. ~50x faster than the per-q Python loop."""
    qs = q_cart if qs is None else qs
    ev = mass_weighted_eigs(dynamical_matrix_batch(qs, unique, K_lab))
    return np.sqrt(np.maximum(ev, 0)) * freq_unit

freqs_bs = bandstructure_freqs(K_LAB_PRIOR)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(x, freqs_bs, color='steelblue', lw=1.2)
for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0],x[-1]); ax.set_ylim(0)
ax.set_title('Rigid-Body Phonon Band Structure — 6o2h (P1), geometric prior')
plt.tight_layout(); plt.savefig('band_structure_prior.png', dpi=150); plt.show()

print(f"Librational rest frequencies at Γ: {freqs_bs[0][3:].round(4)} THz")
print(f"Band range across the path: {freqs_bs[freqs_bs>0].min():.4f} – {freqs_bs.max():.4f} THz")
print("A physically sensible fit should keep this spread modest. A refined model")
print("in which one branch collapses toward zero while others rise by an order of")
print("magnitude is a sign the optimizer has found a boundary, not an optimum.")

## Reading the Experimental Map

Two jobs, one streaming pass. A **diagnostic** sweep — mean intensity, mean
$\sigma$, median $I/\sigma$, finite-voxel coverage and negative fraction as
functions of resolution — and the actual **collection** of candidate fitting
voxels. One $h$-index slice at a time, so peak memory stays at one
$(n_k\times n_l)$ plane regardless of the map's size, and the ~1 GB of `I` and
`sigma` is read once rather than twice.

Two changes from the earlier version are worth naming, because both were doing
real damage.

**The resolution shell is no longer chosen by an SNR heuristic.**
$\overline I/\overline\sigma$ per shell is not an SNR: diffuse intensity rises
steeply with $|q|$ (roughly as $q^2|F|^2$), so the ratio climbs with resolution
mostly because the *numerator* grows, and the heuristic therefore always selects
the finest shell available. That is the worst possible choice here — at ~2 Å the
diffuse scattering from a protein crystal is dominated by internal and side-chain
motion rather than rigid-body lattice dynamics (a central finding of the paper
this data comes from), the one-phonon approximation is degrading, and
Debye-Waller suppression is strongest. `D_FIT_MIN` is now a model-validity
statement set in the parameters cell. The median of $I/\sigma$ is collected as
the honest per-voxel statistic, and $CC$ is reported per resolution shell later
so the resolution at which the model stops working is *measured*.

**The Bragg filter is now a genuine three-index test.** A voxel is Bragg-adjacent
only when it is close to a reciprocal-lattice point in all three indices, tested
on the Cartesian distance. The earlier test, `|dh| < 0.03` in index units against
a map sampled at steps of $1/11$ and $1/13$, could only fire when $h$ was an exact
integer — and then discarded that entire $(k,l)$ plane. Net effect: roughly 9% of
the data thrown away, specifically the planes carrying the strongest halos, while
nothing at all was excluded elsewhere.

In [ ]:
def _bragg_offsets(h_frac, k_frac_all, l_frac_all):
    """Signed offsets to the nearest reciprocal-lattice point, and the Cartesian
    distance to it. Purely geometric -- it knows nothing about I_obs or its sign,
    which is what makes it a legitimate basis for stratifying or weighting.
    Selecting on the sign of the quantity being fit would not be."""
    dh = h_frac - round(h_frac)
    dk = k_frac_all - np.round(k_frac_all)
    dl = l_frac_all - np.round(l_frac_all)
    dq = (dh*B_recip[:,0][None,None,:] + dk[:,:,None]*B_recip[:,1][None,None,:]
          + dl[:,:,None]*B_recip[:,2][None,None,:])
    return dh, dk, dl, np.linalg.norm(dq, axis=-1)

D_SCAN_MAX = 12.0
N_RES_BINS = 40

def scan_and_collect(d_lo, d_hi, n_res_bins=N_RES_BINS):
    """One streaming pass: resolution diagnostics AND candidate-voxel collection.

    Collects everything in the [d_lo, d_hi] shell that is not Bragg-adjacent --
    both the near-Bragg voxels that feed the halo profile and the mid-zone
    voxels that are stratified and subsampled below.
    """
    inv_d_edges = np.linspace(0.0, 2*np.pi/1.0, n_res_bins+1)
    inv_d_mid   = 0.5*(inv_d_edges[:-1] + inv_d_edges[1:])
    sum_I   = np.zeros(n_res_bins); sum_sig = np.zeros(n_res_bins)
    cnt_fin = np.zeros(n_res_bins, np.int64); cnt_tot = np.zeros(n_res_bins, np.int64)
    cnt_neg = np.zeros(n_res_bins, np.int64)
    snr_samples = [[] for _ in range(n_res_bins)]

    q_lo, q_hi = 2*np.pi/d_hi, 2*np.pi/d_lo
    hs, ks, ls, Is, Ss, Bds = [], [], [], [], [], []
    rng_thin = np.random.default_rng(0)

    nH, nK_, nL_ = grid_size
    with h5py.File(H5_FILE, 'r') as f:
        Ids, Sds = f[f'{MAP_GROUP}/I'], f[f'{MAP_GROUP}/sigma']
        jj_idx, kk_idx = np.meshgrid(np.arange(nK_), np.arange(nL_), indexing='ij')
        k_frac_all = grid_ori[1] + jj_idx*grid_delta[1]
        l_frac_all = grid_ori[2] + kk_idx*grid_delta[2]
        for i in range(nH):
            h_frac = grid_ori[0] + i*grid_delta[0]
            q  = (h_frac*B_recip[:,0][None,None,:]
                  + k_frac_all[:,:,None]*B_recip[:,1][None,None,:]
                  + l_frac_all[:,:,None]*B_recip[:,2][None,None,:])
            qn = np.linalg.norm(q, axis=-1)
            I_slice = Ids[i,:,:]; S_slice = Sds[i,:,:]
            finite  = np.isfinite(I_slice) & np.isfinite(S_slice) & (S_slice > 0)

            bin_idx = np.clip(np.digitize(qn, inv_d_edges)-1, 0, n_res_bins-1)
            np.add.at(cnt_tot, bin_idx.ravel(), 1)
            fb = bin_idx[finite]
            np.add.at(cnt_fin, fb, 1)
            np.add.at(sum_I,   fb, I_slice[finite].astype(np.float64))
            np.add.at(sum_sig, fb, S_slice[finite].astype(np.float64))
            np.add.at(cnt_neg, bin_idx[finite & (I_slice < 0)], 1)
            if finite.any():
                r = I_slice[finite]/S_slice[finite]
                thin = rng_thin.random(len(r)) < 0.01
                for b_, v_ in zip(fb[thin], r[thin]):
                    snr_samples[b_].append(float(v_))

            dh, dk, dl, bragg_d = _bragg_offsets(h_frac, k_frac_all, l_frac_all)
            ok = finite & (qn >= q_lo) & (qn <= q_hi) & (bragg_d >= BRAGG_EXCL_CART)
            if not ok.any():
                continue
            sel = np.where(ok)
            hs.append(np.full(len(sel[0]), h_frac))
            ks.append(k_frac_all[sel]); ls.append(l_frac_all[sel])
            Is.append(I_slice[sel].astype(np.float64))
            Ss.append(S_slice[sel].astype(np.float64))
            Bds.append(bragg_d[sel])

    with np.errstate(invalid='ignore', divide='ignore'):
        mean_I     = np.where(cnt_fin > 0, sum_I/np.maximum(cnt_fin,1), np.nan)
        mean_sigma = np.where(cnt_fin > 0, sum_sig/np.maximum(cnt_fin,1), np.nan)
        snr_proxy  = mean_I/mean_sigma
        neg_frac   = np.where(cnt_fin > 0, cnt_neg/np.maximum(cnt_fin,1), np.nan)
    med_snr = np.array([np.median(s) if len(s) > 20 else np.nan for s in snr_samples])
    d_mid   = np.where(inv_d_mid > 1e-6, 2*np.pi/np.maximum(inv_d_mid,1e-6), np.inf)

    hkl = (np.column_stack([np.concatenate(hs), np.concatenate(ks), np.concatenate(ls)])
           if hs else np.zeros((0,3)))
    return (dict(d_mid=d_mid, snr_proxy=snr_proxy, med_snr=med_snr,
                 finite_frac=cnt_fin/np.maximum(cnt_tot,1), neg_frac=neg_frac,
                 cnt_finite=cnt_fin),
            hkl,
            np.concatenate(Is) if Is else np.zeros(0),
            np.concatenate(Ss) if Ss else np.zeros(0),
            np.concatenate(Bds) if Bds else np.zeros(0))

_diag, hkl_pool, I_pool, sigma_pool, bragg_dist_pool = scan_and_collect(D_FIT_MIN, D_FIT_MAX)
d_mid        = _diag['d_mid']
snr_proxy    = _diag['snr_proxy']
med_snr      = _diag['med_snr']
finite_frac  = _diag['finite_frac']
neg_frac_res = _diag['neg_frac']
cnt_finite   = _diag['cnt_finite']

print(f"{len(hkl_pool)} candidate voxels in {D_FIT_MIN}-{D_FIT_MAX} Å "
      f"(one streaming pass; I and sigma each read once)")
print(f"Negative fraction in the candidate pool: {(I_pool < 0).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
finite_d = np.isfinite(d_mid) & (d_mid <= D_SCAN_MAX) & (cnt_finite > 0)

axes[0].plot(d_mid[finite_d], snr_proxy[finite_d], 'o-', ms=3,
             label=r'$\overline{I}/\overline{\sigma}$ (misleading)')
axes[0].plot(d_mid[finite_d], med_snr[finite_d], 's-', ms=3, color='darkgreen',
             label=r'median $I/\sigma$ (honest)')
axes[0].axhline(1.0, color='r', ls='--', lw=1)
axes[0].axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.2, label='fitting range')
axes[0].set_xlabel('resolution $d$ (Å)'); axes[0].set_ylabel('signal-to-noise')
axes[0].invert_xaxis(); axes[0].legend(fontsize=8)
axes[0].set_title('Signal-to-noise vs. resolution')

axes[1].plot(d_mid[finite_d], finite_frac[finite_d], 'o-', ms=3, color='darkorange')
axes[1].axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.2)
axes[1].set_xlabel('resolution $d$ (Å)'); axes[1].set_ylabel('finite-voxel fraction')
axes[1].invert_xaxis(); axes[1].set_title('Data coverage vs. resolution')

axes[2].plot(d_mid[finite_d], neg_frac_res[finite_d], 'o-', ms=3, color='crimson')
axes[2].axhline(0.5, color='k', ls=':', lw=1)
axes[2].axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.2)
axes[2].set_xlabel('resolution $d$ (Å)'); axes[2].set_ylabel(r'fraction with $I<0$')
axes[2].invert_xaxis(); axes[2].set_title('Negative-intensity fraction vs. resolution')

plt.tight_layout(); plt.savefig('resolution_snr_scan.png', dpi=150); plt.show()

print(f"Fitting range (set by model validity, not by SNR): {D_FIT_MIN}-{D_FIT_MAX} Å")
_in = finite_d & (d_mid >= D_FIT_MIN) & (d_mid <= D_FIT_MAX)
if _in.any():
    print(f"  median I/σ there:        {np.nanmin(med_snr[_in]):.2f} – {np.nanmax(med_snr[_in]):.2f}")
    print(f"  negative fraction there: {np.nanmin(neg_frac_res[_in]):.1%} – "
          f"{np.nanmax(neg_frac_res[_in]):.1%}")
print("\nNote how the two SNR curves diverge toward high resolution: that gap is")
print("exactly the artefact that made the old shell-selection heuristic pick the")
print("finest bin available.")

## Negative Intensities: What They Do and Do Not Break

A substantial fraction of voxels carry $I_{\rm obs}<0$, because Compton and
detector-geometry corrections are subtracted during processing and noise pushes
weak pixels below zero. Three statements, in order of how often they get confused:

1. **Least squares against negative data is not ill-behaved.** If
   $I_{\rm obs}=I_{\rm true}+\varepsilon$ with $\mathbb E[\varepsilon]=0$, then
   minimizing $\sum w(I_{\rm model}-I_{\rm obs})^2$ is a *consistent* estimator of
   $I_{\rm true}$ even where most observations are negative. Individual residuals
   are large but unbiased, and $1/\sigma^2$ already down-weights them correctly.
   Nothing needs fixing about the estimator itself.

2. **Filtering on the sign of $I_{\rm obs}$ would be a real error.** Keeping only
   voxels where noise pushed $I_{\rm obs}$ above zero while discarding the equally
   informative ones it pushed below inflates the retained sample's mean above the
   truth. It would also mechanically improve $R$/$R_{\rm free}$ without the fit
   being more accurate, since the points guaranteed to contribute large residuals
   are exactly the ones removed. Negative points are kept throughout.

3. **Negatives are a symptom, not the cause.** A high negative fraction means
   SNR $\lesssim1$. The danger is not the sign but that such voxels carry weight
   and no information — and because $I_{\rm model}=G^\dagger D^{-1}G\ge0$ always,
   the least-squares optimum for a region whose weighted mean is $\approx0$ is
   $I_{\rm model}\to0$, which the model reaches by driving $K$ stiff. **That is
   the runaway mechanism**, and its signature is $R\to1$ *together with*
   $CC\to0$: a model predicting nothing, which $R$ alone cannot distinguish from
   a merely poor fit. Both are reported everywhere below.

So the instinct to fit the halos is right, but for the signal-to-noise reason
rather than the sign reason — and the response is to fit where there is signal
and to report the effective sample size, not to censor the data.

### Halo Profile and Stratified Mid-Zone Sampling

Two complementary sets of observations, for two different parts of the model.

**Halo profile.** $I_{\rm obs}$ averaged over (radial shell × angular sector)
bins of the offset from the nearest reciprocal-lattice point, pooled across all
such points in range. This beats the noise down by $\sim\sqrt N$, turning
$\sim10^6$ noisy voxels into a few dozen well-determined numbers, and the
$I\sim|\mathbf q-\mathbf G|^{-2}$ shape with its anisotropic prefactor is a
direct readout of the long-wavelength limit of $D(\mathbf q)$ — the acoustic and
elastic information, which is the part this data can genuinely determine.
Sectoring rather than pure radial averaging is what preserves the anisotropy,
and the anisotropy is where the information about *which* contacts are stiff
lives. As a bonus the shell averages are positive wherever there is real signal,
so the negative-intensity problem simply does not arise for this term.

**Stratified mid-zone sample.** Equal-count quotas over a (resolution ×
Bragg-distance) grid of strata, then pure inverse-variance weights.

This replaces a multiplicative Gaussian halo weight applied to a uniformly-drawn
pool, and the difference is not cosmetic. Because the number of voxels at Bragg
distance $d$ grows as $d^2$, a uniform pool is dominated by large $d$; a Gaussian
factor with a short length scale then *annihilates* most of it rather than
reweighting it. A nominal 1292 training points can carry an effective sample size
of a couple of hundred **before** the $1/\sigma^2$ spread is applied — not enough
to determine 126 parameters, and enough on its own to produce a degenerate,
boundary-seeking $K$. Expressing the halo preference through *sampling* instead
means every retained point actually counts.

The effective sample size $(\sum w)^2/\sum w^2$ is printed next to every nominal
count from here on, because a nominal count can hide exactly this failure.

In [ ]:
def effective_sample_size(w):
    """ESS = (Σw)²/Σw². A nominal N means nothing once weights are heterogeneous."""
    w = np.asarray(w, float)
    return w.sum()**2/np.sum(w**2)

def make_weights(sigma):
    """Inverse-variance weights with a percentile floor on sigma, so a handful of
    anomalously small error bars cannot dominate the whole objective."""
    s = np.asarray(sigma, float)
    floor = np.percentile(s[np.isfinite(s) & (s > 0)], SIGMA_FLOOR_PCT)
    return 1.0/np.maximum(s, floor)**2

# --- Diagnostic: is the negative-intensity pattern really halo-related? ----
nbins = 12
bd_edges = np.linspace(0, np.percentile(bragg_dist_pool, 99), nbins+1)
bd_cen   = 0.5*(bd_edges[:-1]+bd_edges[1:])
bd_idx   = np.clip(np.digitize(bragg_dist_pool, bd_edges)-1, 0, nbins-1)
neg_frac_bd = np.array([(I_pool[bd_idx==b] < 0).mean() if (bd_idx==b).any() else np.nan
                        for b in range(nbins)])
mean_I_bd   = np.array([I_pool[bd_idx==b].mean() if (bd_idx==b).any() else np.nan
                        for b in range(nbins)])
cnt_bd      = np.array([(bd_idx==b).sum() for b in range(nbins)])

fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(bd_cen, neg_frac_bd, 'o-', color='crimson')
axes[0].set_xlabel(r'distance to nearest Bragg point (Å$^{-1}$)')
axes[0].set_ylabel('fraction with $I_{obs}<0$'); axes[0].set_title('Negative fraction')
axes[1].plot(bd_cen, mean_I_bd, 'o-', color='steelblue')
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'distance to nearest Bragg point (Å$^{-1}$)')
axes[1].set_ylabel(r'mean $I_{obs}$'); axes[1].set_title('Mean intensity')
axes[2].semilogy(bd_cen, np.maximum(cnt_bd,1), 'o-', color='darkgreen')
axes[2].set_xlabel(r'distance to nearest Bragg point (Å$^{-1}$)')
axes[2].set_ylabel('voxel count'); axes[2].set_title(r'Population grows as $d^2$')
plt.tight_layout(); plt.savefig('bragg_distance_diagnostic.png', dpi=150); plt.show()

print("The right-hand panel is why a multiplicative halo weight fails: the pool is")
print("dominated by large Bragg distance, so a short-length-scale Gaussian weight")
print("discards most of the sample rather than reweighting it.")

# --- Halo profile ---------------------------------------------------------
def build_halo_profile(hkl, I, sigma, bragg_d, r_max=HALO_R_MAX,
                       n_shell=N_HALO_SHELLS, n_dir=N_HALO_DIRS, min_count=30):
    """Weighted mean I in (radial shell × angular sector) bins of the offset from
    the nearest reciprocal-lattice point, pooled over all such points.

    Returns representative (h,k,l) -- the weighted centroid of each bin -- so the
    profile feeds the same forward model as the mid-zone points, together with
    the bin mean and its standard error.
    """
    if len(hkl) == 0:
        return np.zeros((0,3)), np.zeros(0), np.zeros(0), np.zeros(0, int)
    q  = hkl @ B_recip.T
    dq = q - (np.round(hkl) @ B_recip.T)
    sel = (bragg_d > BRAGG_EXCL_CART) & (bragg_d < r_max)
    if sel.sum() == 0:
        return np.zeros((0,3)), np.zeros(0), np.zeros(0), np.zeros(0, int)
    hkl_s, I_s, sig_s, bd_s, dq_s = hkl[sel], I[sel], sigma[sel], bragg_d[sel], dq[sel]
    u = dq_s/np.linalg.norm(dq_s, axis=1, keepdims=True)
    theta = np.arccos(np.clip(u[:,2], -1, 1)); phi = np.arctan2(u[:,1], u[:,0])
    n_th = max(1, int(round(np.sqrt(n_dir)))); n_ph = max(1, n_dir//n_th)
    a_idx = (np.clip((theta/np.pi*n_th).astype(int), 0, n_th-1)*n_ph
             + np.clip(((phi+np.pi)/(2*np.pi)*n_ph).astype(int), 0, n_ph-1))
    r_edges = np.linspace(BRAGG_EXCL_CART, r_max, n_shell+1)
    r_idx = np.clip(np.digitize(bd_s, r_edges)-1, 0, n_shell-1)
    key = r_idx*(n_th*n_ph) + a_idx
    w = make_weights(sig_s)
    pts, mean, err, cnt = [], [], [], []
    for k_ in np.unique(key):
        m = key == k_
        if m.sum() < min_count:
            continue
        W = w[m].sum()
        pts.append(np.average(hkl_s[m], axis=0, weights=w[m]))
        mean.append(float(np.sum(w[m]*I_s[m])/W))
        err.append(float(1.0/np.sqrt(W)))
        cnt.append(int(m.sum()))
    if not pts:
        return np.zeros((0,3)), np.zeros(0), np.zeros(0), np.zeros(0, int)
    return np.array(pts), np.array(mean), np.array(err), np.array(cnt)

hkl_halo, I_halo, sig_halo, cnt_halo = build_halo_profile(
    hkl_pool, I_pool, sigma_pool, bragg_dist_pool)
print(f"\nHalo profile: {len(hkl_halo)} (shell × sector) bins from {cnt_halo.sum()} voxels")
if len(hkl_halo):
    print(f"  {cnt_halo.min()}-{cnt_halo.max()} voxels per bin;  "
          f"all bin means positive: {bool((I_halo > 0).all())}")
    print("  (shell averaging removes the negative-intensity problem for this term)")

# --- Stratified mid-zone sample -------------------------------------------
rng = np.random.default_rng(7)
q_norm_pool = np.linalg.norm(hkl_pool @ B_recip.T, axis=1)
idx_all = np.where(bragg_dist_pool >= HALO_R_MAX)[0]

q_bins = np.quantile(q_norm_pool[idx_all], np.linspace(0,1,N_RES_STRATA+1))
b_bins = np.quantile(bragg_dist_pool[idx_all], np.linspace(0,1,N_HALO_STRATA+1))
qi = np.clip(np.digitize(q_norm_pool[idx_all], q_bins)-1, 0, N_RES_STRATA-1)
bi = np.clip(np.digitize(bragg_dist_pool[idx_all], b_bins)-1, 0, N_HALO_STRATA-1)
strata_key = qi*N_HALO_STRATA + bi
per = max(1, N_FIT_MAX//(N_RES_STRATA*N_HALO_STRATA))
picked = [rng.choice(idx_all[strata_key==s_], size=min(per, int((strata_key==s_).sum())),
                     replace=False)
          for s_ in range(N_RES_STRATA*N_HALO_STRATA) if (strata_key==s_).any()]
sub_idx = np.concatenate(picked)

hkl_mid, I_mid, sigma_mid = hkl_pool[sub_idx], I_pool[sub_idx], sigma_pool[sub_idx]
train_mask = rng.random(len(sub_idx)) > HOLDOUT_FRAC

q_train, I_train, sigma_train = hkl_mid[train_mask], I_mid[train_mask], sigma_mid[train_mask]
q_test,  I_test,  sigma_test  = hkl_mid[~train_mask], I_mid[~train_mask], sigma_mid[~train_mask]
w_train_mid = make_weights(sigma_train)

print(f"\nMid-zone sample: {len(sub_idx)} points over {N_RES_STRATA}×{N_HALO_STRATA} strata")
print(f"  training {train_mask.sum()} (effective {effective_sample_size(w_train_mid):.0f}), "
      f"held out {(~train_mask).sum()}")
print(f"  negative fraction in training: {(I_train < 0).mean():.1%} "
      f"(kept -- filtering on sign would bias the fit)")

In [ ]:
def well_conditioned_batch(K_lab, hkl, min_eig_frac=1e-4):
    """D(q) is exactly singular only at reciprocal-lattice points, but a point can
    still land near a degenerate direction, where the plain linear solve the
    objective uses is ill-conditioned.

    Evaluated at the PRIOR K -- i.e. at the actual operating point the refinement
    starts from -- rather than at an unrelated placeholder, and vectorized.
    """
    if len(hkl) == 0:
        return np.zeros(0, bool)
    q = (hkl[:,0:1]*B_recip[:,0] + hkl[:,1:2]*B_recip[:,1] + hkl[:,2:3]*B_recip[:,2])
    ev = np.linalg.eigvalsh(dynamical_matrix_batch(q, unique, K_lab))
    emax, emin = ev[:,-1], ev[:,0]
    return (emax > 0) & (emin/np.where(emax > 0, emax, 1.0) > min_eig_frac)

keep_tr = well_conditioned_batch(K_LAB_PRIOR, q_train)
keep_te = well_conditioned_batch(K_LAB_PRIOR, q_test)
keep_ha = well_conditioned_batch(K_LAB_PRIOR, hkl_halo)
print(f"conditioning screen: training {keep_tr.sum()}/{len(keep_tr)}, "
      f"held out {keep_te.sum()}/{len(keep_te)}, halo {keep_ha.sum()}/{len(keep_ha)}")

q_train, I_train, sigma_train = q_train[keep_tr], I_train[keep_tr], sigma_train[keep_tr]
q_test,  I_test,  sigma_test  = q_test[keep_te],  I_test[keep_te],  sigma_test[keep_te]
hkl_halo, I_halo, sig_halo, cnt_halo = (hkl_halo[keep_ha], I_halo[keep_ha],
                                        sig_halo[keep_ha], cnt_halo[keep_ha])
w_train_mid = make_weights(sigma_train)

### Assembling the Objective

The halo bins and the mid-zone points enter the *same* weighted least-squares sum.
Each halo bin carries the standard error of its own average, so its natural weight
is $1/\mathrm{SE}^2$ — large, as it should be, since each bin condenses thousands
of voxels. The halo block is then renormalized so the two terms carry comparable
total weight before `HALO_WEIGHT` scales it, otherwise the balance between them
would depend on how many bins the profile happened to produce. Set `HALO_WEIGHT`
to 0 to fit the mid-zone alone, or raise it to lean harder on the acoustic limit.

In [ ]:
if len(hkl_halo):
    w_halo = HALO_WEIGHT/np.maximum(sig_halo, 1e-12)**2
    w_halo = w_halo*(w_train_mid.sum()/max(w_halo.sum(), 1e-30))
    q_fit = np.vstack([q_train, hkl_halo])
    I_fit = np.concatenate([I_train, I_halo])
    w_fit = np.concatenate([w_train_mid, w_halo])
    is_halo = np.concatenate([np.zeros(len(q_train), bool), np.ones(len(hkl_halo), bool)])
else:
    w_halo = np.zeros(0)
    q_fit, I_fit, w_fit = q_train, I_train, w_train_mid
    is_halo = np.zeros(len(q_train), bool)

ess = effective_sample_size(w_fit)
print(f"Objective: {len(q_fit)} observations "
      f"({len(q_train)} mid-zone + {len(hkl_halo)} halo bins)")
print(f"  effective sample size: {ess:.0f}")
print(f"  weight share: mid-zone {100*w_train_mid.sum()/w_fit.sum():.0f}%, "
      f"halo {100*w_halo.sum()/w_fit.sum() if len(w_halo) else 0:.0f}%")
print(f"  free parameters: {N_PARAMS} (the shape normalization removes 1 of them)")
if ess < 5*N_PARAMS:
    print("\n*** WARNING: fewer than 5 effective observations per parameter. Expect")
    print("    the prior to dominate; read the sloppiness spectrum before quoting K. ***")

$G(\mathbf q)$ depends only on the structure and $\mathbf q$, never on the
contact stiffnesses being fit, so it is computed **once** for the fitting and
held-out sets rather than recomputed from scratch at every refinement iteration.
With L-BFGS-B running to hundreds or thousands of iterations, that turns thousands
of redundant transform evaluations into one.

In [ ]:
G_fit  = G_at_hkl_batch(q_fit[:,0],  q_fit[:,1],  q_fit[:,2])
G_test = (G_at_hkl_batch(q_test[:,0], q_test[:,1], q_test[:,2])
          if len(q_test) else np.zeros((0,6), complex))
print(f"Cached G(q) for {len(G_fit)} fitting and {len(G_test)} held-out points")

### Two Intensity Code Paths, Asserted Against Each Other

The refinement objective evaluates $I=G^\dagger D^{-1}G$ with a batched linear
solve; the maps and the Brillouin-zone integral use an eigendecomposition with a
pseudo-inverse. Those must agree, and the assertion below enforces it.

This is not a formality. If either path silently takes a real part — of $G$, or
of $D$ — the figures and the predicted ADPs would be showing a *different model*
from the one that was fit, at a median factor of roughly $0.46$ in $I(\mathbf q)$
with a five-fold spread. A synthetic ground-truth test cannot catch that class of
error, because it compares two quantities computed identically-wrongly and the
error cancels. An explicit cross-path assertion can.

In [ ]:
def diffuse_intensity_batch(K_lab, h_arr, k_arr, l_arr, chunk=None,
                            min_eig_frac=1e-6, G_arr=None):
    """I(q) = G^† D(q)^+ G, vectorized.

    FULL COMPLEX G and D. D is periodic in the reciprocal lattice, so D(G)=D(0)
    at every Bragg position: the 3 acoustic eigenvalues vanish there and a
    pseudo-inverse keeps I finite. Immediately off a Bragg spot the acoustic
    eigenvalues grow as |δq|², producing the I ~ |q-G|^-2 halos.
    """
    chunk = chunk or CHUNK_EVAL
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    N = len(h_arr); I_out = np.zeros(N)
    for s in range(0, N, chunk):
        sl = slice(s, min(s+chunk, N))
        qb = (h_arr[sl,None]*B_recip[:,0] + k_arr[sl,None]*B_recip[:,1]
              + l_arr[sl,None]*B_recip[:,2])
        Db = dynamical_matrix_batch(qb, unique, K_lab)
        Gb = G_arr[sl] if G_arr is not None else G_at_hkl_batch(h_arr[sl], k_arr[sl], l_arr[sl])
        ev, evec = np.linalg.eigh(Db)                      # complex Hermitian
        emax = np.clip(ev.max(axis=1, keepdims=True), 1e-15, None)
        pos = ev > (min_eig_frac*emax)
        ev_inv = np.where(pos, 1.0/np.where(pos, ev, 1.0), 0.0)
        proj = np.einsum('nji,nj->ni', evec.conj(), Gb) * ev_inv
        DinvG = np.einsum('nij,nj->ni', evec, proj)
        I_out[sl] = np.real(np.einsum('ni,ni->n', Gb.conj(), DinvG))
    return I_out

def forward_I(K_lab, h, k, l, G=None):
    """Single-point I(q) by a plain (non-pseudo-inverse) solve -- used where
    D(q) is known to be comfortably invertible, i.e. away from Bragg points."""
    q = h*B_recip[:,0] + k*B_recip[:,1] + l*B_recip[:,2]
    if G is None:
        G = G_at_hkl(h, k, l)
    D = dynamical_matrix(q, unique, K_lab)
    return float(np.real(np.conj(G) @ np.linalg.solve(D, G)))

# CONSISTENCY CHECK -- the guard that a synthetic-only test cannot provide.
# The refinement objective uses forward_I's solve path; the maps and the ADP
# integral use diffuse_intensity_batch's eigendecomposition path. If those two
# ever disagree, one of them has silently dropped an imaginary part and the
# figures will be showing a different model from the one being fit.
_hq = np.array([1.37, -2.13, 0.61, 3.05, -1.44])
_kq = np.array([0.22,  1.81, -2.4, 0.13,  2.77])
_lq = np.array([0.45, -0.33, 1.12, -2.06, 0.88])
_Ia = diffuse_intensity_batch(K_LAB_PRIOR, _hq, _kq, _lq)
_Ib = np.array([forward_I(K_LAB_PRIOR, h, k, l) for h, k, l in zip(_hq, _kq, _lq)])
_rel = np.abs(_Ia - _Ib)/np.maximum(np.abs(_Ib), 1e-30)
print(f"Batched vs. single-point I(q): max relative difference {_rel.max():.3e}")
assert _rel.max() < 1e-8, "the two intensity code paths disagree -- see markdown"
print("OK: both intensity paths agree, so the plotted model is the fitted model.")


## Where the Fitted q-Points Sit

The left panel shows all fitting points in true Cartesian $\mathbf q$; the two
right panels overlay them on the **actual experimental map** at $l=0$ and
$l\approx0.5$ (single 2-D slices read straight from the HDF5 file — cheap
regardless of the full map's size).

The overlay selects a **slab** of finite thickness in $l$ rather than an exact
layer. An exact-match tolerance picked out a handful of points from thousands and
produced a nearly content-free figure; the slab shows where the sampling actually
sits relative to the halos.

In [ ]:
def nearest_l_index(l_target):
    return int(round((l_target - grid_ori[2])/grid_delta[2]))

def read_hk_plane(l_index):
    with h5py.File(H5_FILE, 'r') as f:
        I_plane = f[f'{MAP_GROUP}/I'][:, :, l_index]
    h_pts = grid_ori[0] + np.arange(grid_size[0])*grid_delta[0]
    k_pts = grid_ori[1] + np.arange(grid_size[1])*grid_delta[1]
    return h_pts, k_pts, np.asarray(I_plane)

l_idx_0    = nearest_l_index(0.0)
l_idx_half = nearest_l_index(0.5)
h_pts_bg0,  k_pts_bg0,  I_bg0  = read_hk_plane(l_idx_0)
h_pts_bg05, k_pts_bg05, I_bg05 = read_hk_plane(l_idx_half)
l_layer_0  = grid_ori[2] + l_idx_0*grid_delta[2]
l_layer_05 = grid_ori[2] + l_idx_half*grid_delta[2]

SLAB = 4*grid_delta[2]      # slab half-thickness for the overlay

qc_fit  = q_fit  @ B_recip.T
qc_test = q_test @ B_recip.T

fig = plt.figure(figsize=(16, 5))
ax3d = fig.add_subplot(1, 3, 1, projection='3d')
ax3d.scatter(*qc_fit[~is_halo].T, s=6, alpha=0.4, color='steelblue', label='mid-zone')
if is_halo.any():
    ax3d.scatter(*qc_fit[is_halo].T, s=22, alpha=0.9, color='limegreen', label='halo bins')
ax3d.scatter(*qc_test.T, s=6, alpha=0.5, color='orangered', label='held out')
ax3d.set_xlabel('$q_x$'); ax3d.set_ylabel('$q_y$'); ax3d.set_zlabel('$q_z$')
ax3d.set_title('Sampled q-points (Å$^{-1}$)'); ax3d.legend(fontsize=8)

def overlay_panel(ax, h_pts, k_pts, I_bg, l_layer):
    pos = I_bg[np.isfinite(I_bg) & (I_bg > 0)]
    vmax = np.percentile(pos, 97) if len(pos) else 1.0
    I_disp = np.log1p(np.clip(np.nan_to_num(I_bg, nan=0.0), 0, vmax))/np.log(10)
    ax.imshow(I_disp.T, origin='lower', cmap='gray',
              extent=[h_pts[0], h_pts[-1], k_pts[0], k_pts[-1]], aspect='auto', alpha=0.85)
    for pts, color, label in [(q_fit, 'steelblue', 'fitting'), (q_test, 'orangered', 'held out')]:
        sel = np.abs(pts[:,2] - l_layer) < SLAB
        ax.scatter(pts[sel,0], pts[sel,1], s=8, color=color, edgecolor='none',
                   alpha=0.6, label=f'{label} ({sel.sum()})')
    ax.set_xlabel('h'); ax.set_ylabel('k')
    ax.set_title(f'l ≈ {l_layer:.3f} ± {SLAB:.3f}  (experimental map)')
    ax.legend(fontsize=7, loc='upper right')

overlay_panel(fig.add_subplot(1,3,2), h_pts_bg0,  k_pts_bg0,  I_bg0,  l_layer_0)
overlay_panel(fig.add_subplot(1,3,3), h_pts_bg05, k_pts_bg05, I_bg05, l_layer_05)
plt.tight_layout(); plt.savefig('sampled_q_points.png', dpi=150); plt.show()

## Refining $K$ to the Experimental Data

$$\mathcal L(\theta,s)=\sum_{\mathbf q} w(\mathbf q)
\bigl[s\,I_{\rm model}(\mathbf q;\theta)-I_{\rm obs}(\mathbf q)\bigr]^2
+\lambda\sum_{\mathbf n}
\frac{\|K_{\mathbf n}-K_{\mathbf n}^{\rm prior}\|_F^2}{\|K_{\mathbf n}^{\rm prior}\|_F^2}$$

with $w=1/\sigma_{\rm exp}^2$ taken from the deposited map's own per-voxel error
estimate (floored at a low percentile), and no noise model assumed anywhere.

### The $K$/scale degeneracy, and why the ADPs must stay out of it

$D$ is linear in $K$, so $K\to\lambda K$ sends $I_{\rm model}\to I_{\rm model}/\lambda$
exactly — and the predicted ADPs rescale the same way. With $s$ free, the
combination $\lambda s$ is therefore **exactly** unidentifiable from the diffuse
map. Two things were previously tried and both fail:

* **Profile $s$ freely at every iteration.** Mathematically consistent, but it
  makes the data loss exactly invariant to the overall magnitude of $K$, leaving
  that direction controlled only by the regularizer — which prefers smaller $K$.
  Refinement then runs away toward implausibly soft contacts.
* **Pin $s$ once, from an arbitrary starting $K$.** This does not remove the
  degeneracy; it picks an arbitrary point on it. The data then constrains $K$ only
  relative to a number chosen in advance, and the fit lands wherever that number
  put it.

A third option — rescaling the fitted $K$ afterwards so the predicted mean $B$
matches the deposited mean — is worse than either, for two reasons. It makes the
headline validation a tautology (the means match because they were defined to).
And it is guaranteed to over-soften $K$ even if everything else were perfect,
because the deposited $B$ contains internal and substitutional disorder on top of
the lattice motion, so the lattice model should predict *less*.

**What is done instead.** Remove the degenerate direction from the parameter
space: refine the normalized **shape** of $K$ (with $\sum_{\mathbf n}\mathrm{tr}\,
K_{\mathbf n}$ held fixed), profiling $s$ in closed form each call. There is then
no runaway direction at all, and the regularizer is back to being a mild bias.
Afterwards, close the gauge with **units**: with $k_BT=1$ and $G$ in $e/\mathrm{Å}$,
$I_{\rm model}$ is in electrons² per unit cell, the same scale the deposited map
reports, so $s\equiv1$ is the physical statement. That fixes $K$ absolutely using
zero information from the ADPs, leaving the predicted $B$ a genuine prediction.

The profiled $s^\star$ at the fitted shape is itself worth reporting: if it comes
out far from 1, the discrepancy is a real result about the model (missing
non-lattice scattering, multi-phonon terms, or a normalization difference in the
deposited map), not something to absorb silently.

**Gradient.** $dI=-v^\dagger dD\,v$ with $v=D^{-1}G$, and $D$ is linear in each
$K_{\mathbf n}$, so $dI/dK_{\mathbf n}$ is closed-form at essentially no cost
beyond the forward evaluation. Chain through the fixed lab-frame transform and
$K=LL^{\mathsf T}$ for $d\mathcal L/dL$ (verified below against finite
differences). When $s$ is profiled, the envelope theorem means no extra term is
needed for its dependence on $\theta$.

**Optimizer.** L-BFGS-B, fully vectorized over the observation batch. No box
bounds are needed: the earlier `L_BOUND` safety rail existed to stop the runaway
described above, and with the degenerate direction removed and the prior
regularizer in place there is nothing for it to catch. Convergence is
**asserted**, not assumed.

In [ ]:
def normalize_shape(Lmap, target=None):
    """Project onto the fixed-normalization shape manifold: Σ_n tr(K_n) = target.
    This removes the single degenerate direction from the parameter space, so the
    overall magnitude of K is carried by the profiled scale s alone and is not a
    free parameter competing with it."""
    if target is None:
        target = 6.0*len(Lmap)
    tot = sum(np.trace(L @ L.T) for L in Lmap.values())
    f = np.sqrt(target/tot)
    return {c: f*L for c, L in Lmap.items()}

def _model_raw(theta, q_arr, G_arr):
    """I_model before the overall scale, plus the solved v = D^-1 G and the
    Cartesian q batch (both reused by the gradient)."""
    Lmap = theta_to_Lmap(theta)
    K_local, K_lab = build_K(Lmap)
    q_batch = (q_arr[:,0:1]*B_recip[:,0] + q_arr[:,1:2]*B_recip[:,1]
               + q_arr[:,2:3]*B_recip[:,2])
    D_batch = dynamical_matrix_batch(q_batch, unique, K_lab)
    V = np.linalg.solve(D_batch, G_arr[:,:,None])[:,:,0]
    return np.real(np.sum(np.conj(G_arr)*V, axis=1)), V, q_batch, Lmap, K_local

def profiled_scale(theta, q_arr, G_arr, I_obs, weights):
    """Closed-form weighted least-squares scale. Valid via the envelope theorem;
    exposed standalone so it can be recomputed once from the FIXED fitted shape
    for reporting, always from the fitting set, never from the held-out set."""
    Iraw, _, _, _, _ = _model_raw(theta, np.asarray(q_arr, float), np.asarray(G_arr))
    denom = np.sum(weights*Iraw**2)
    return float(np.sum(weights*Iraw*I_obs)/denom) if denom > 0 else 1.0

def loss_and_grad(theta, q_arr, G_arr, I_obs, weights):
    """Weighted least squares with s profiled in closed form, plus the
    prior-centred regularizer. Vectorized over the whole observation batch."""
    q_arr = np.asarray(q_arr, float); G_arr = np.asarray(G_arr)
    I_obs = np.asarray(I_obs); weights = np.asarray(weights)
    Iraw, V, q_batch, Lmap, K_local = _model_raw(theta, q_arr, G_arr)

    denom = np.sum(weights*Iraw**2)
    s = float(np.sum(weights*Iraw*I_obs)/denom) if denom > 0 else 1.0
    resid = s*Iraw - I_obs
    data_loss = float(np.sum(weights*resid**2))
    coeff = 2.0*weights*resid*s     # envelope theorem: ds/dtheta contributes nothing

    N_mat = {c: Gc_inv_matrix(unique[c]['c_local']) @ unique[c]['Gamma'].T
             for c in shell_order}
    grad_L = {}
    for c in shell_order:
        R_n = unique[c]['R_n']
        A_n = ad_translation(R_n)
        phase = np.exp(1j*(q_batch @ R_n))
        U  = V @ A_n.T
        w1 = U - phase[:,None]*V
        w2 = V - phase.conj()[:,None]*U
        Graw = -(np.einsum('ni,nj->nij', U.conj(), w1) + np.einsum('ni,nj->nij', V.conj(), w2))
        G_A  = np.einsum('ij,njk,lk->nil', N_mat[c], Graw, N_mat[c])
        dL_c = np.real(np.einsum('nij,jk->nik', G_A + G_A.transpose(0,2,1), Lmap[c]))
        grad_L[c] = np.einsum('n,nij->ij', coeff, dL_c)

    reg  = regularizer_prior(K_local)
    dRdK = regularizer_prior_dK(K_local)
    for c in shell_order:
        grad_L[c] += (dRdK[c] + dRdK[c].T) @ Lmap[c]
        grad_L[c] = np.tril(grad_L[c])

    return data_loss + reg, Lmap_to_theta(grad_L)

In [ ]:
# Gradient check: analytic vs. central finite differences. The scale is profiled
# inside loss_and_grad, so this also exercises the envelope-theorem term being
# correctly absent.
rng_chk = np.random.default_rng(42)
theta_chk = Lmap_to_theta(normalize_shape(L_PRIOR))*(1 + 0.05*rng_chk.standard_normal(N_PARAMS))
n_chk = min(40, len(q_fit))
q_chk, G_chk, I_chk, w_chk = q_fit[:n_chk], G_fit[:n_chk], I_fit[:n_chk], w_fit[:n_chk]

loss0, grad0 = loss_and_grad(theta_chk, q_chk, G_chk, I_chk, w_chk)
eps, errs = 1e-6, []
for idx in rng_chk.choice(N_PARAMS, size=min(8, N_PARAMS), replace=False):
    tp = theta_chk.copy(); tp[idx] += eps
    tm = theta_chk.copy(); tm[idx] -= eps
    fd = (loss_and_grad(tp, q_chk, G_chk, I_chk, w_chk)[0]
          - loss_and_grad(tm, q_chk, G_chk, I_chk, w_chk)[0])/(2*eps)
    errs.append(abs(fd - grad0[idx])/max(abs(fd), 1e-6))
print(f"Gradient check, max relative error over {len(errs)} random directions: {max(errs):.2e}")
assert max(errs) < 1e-3

In [ ]:
MAXITER = 5000

theta0 = Lmap_to_theta(normalize_shape(L_PRIOR))
theta_history = [theta0.copy()]

result = minimize(loss_and_grad, theta0, args=(q_fit, G_fit, I_fit, w_fit),
                  jac=True, method='L-BFGS-B',
                  options={'maxiter': MAXITER, 'ftol': 1e-14, 'gtol': 1e-12},
                  callback=lambda xk: theta_history.append(xk.copy()))

print(result.message)
print(f"iterations: {result.nit}   final loss: {result.fun:.6g}   "
      f"||grad||: {np.linalg.norm(result.jac):.3e}")
if result.nit >= MAXITER:
    print("\n*** NOT CONVERGED: stopped on maxiter. Do not quote these parameters. ***")

theta_fit = result.x
_, K_LAB_SHAPE = build_K(theta_to_Lmap(theta_fit))

# --- Close the gauge with UNITS, not with the ADPs -------------------------
# I_model = G^dag D^-1 G with k_BT = 1 and G in e/Å is in electrons² per unit
# cell, the same scale the deposited map reports. So the physical statement is
# s == 1, and absorbing the profiled s into K achieves exactly that. This uses
# no information from the deposited B-factors, which is what keeps the ADP
# comparison an independent check.
S_STAR = profiled_scale(theta_fit, q_fit, G_fit, I_fit, w_fit)
K_LAB_FIT = {c: S_STAR*K_LAB_SHAPE[c] for c in K_LAB_SHAPE}
S_FIT = 1.0

print(f"\nProfiled scale at the fitted shape: s* = {S_STAR:.4g}")
print(f"  K_LAB_FIT = s* × K_shape, so the effective scale is now exactly 1.")
print(f"  If |log10 s*| is large, that is a REAL statement about the model --")
print(f"  missing non-lattice scattering, multi-phonon terms, or a normalization")
print(f"  difference in the deposited map -- and should be reported, not absorbed.")

chi2_dof = result.fun/max(len(q_fit) - N_PARAMS, 1)
print(f"\nweighted χ²/dof ≈ {chi2_dof:.4f}")

## Watching the Band Structure Converge

There is no ground-truth $K$ here — the crystal's phonon dispersion is what the
data is being used to infer — so the thick grey reference line is the *final*
refined fit and the coloured line is the current iterate.

Two things to watch for, both diagnostic rather than decorative. First, the
acoustic branches near Γ should lock on quickly while the librational branches
drift: that asymmetry is the visible face of the sloppiness quantified below, and
its absence would be suspicious. Second, and more important, a branch that
**collapses toward zero** or one that rises by an order of magnitude relative to
the prior is a sign the optimizer has walked to a boundary of the search space
rather than an interior optimum — the symptom that motivated the prior-centred
regularizer and the shape normalization in the first place. A warning is printed
below if that happens.

In [ ]:
freqs_final_bs = bandstructure_freqs(K_LAB_FIT)
freqs_prior_bs = bandstructure_freqs(K_LAB_PRIOR)
ymax = max(freqs_final_bs.max(), freqs_prior_bs.max())*1.15

frame_idx = np.unique(np.linspace(0, len(theta_history)-1,
                                  min(60, len(theta_history))).astype(int))
legend_handles = [Line2D([0],[0], color='0.6', lw=3.5, label='Final refined fit'),
                  Line2D([0],[0], color='steelblue', lw=1.3, label='Current iteration')]

images_bs = []
for fi in frame_idx:
    _, K_lab_fi = build_K(theta_to_Lmap(theta_history[fi]))
    freqs_fi = bandstructure_freqs(K_lab_fi)*S_STAR**0.5
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, freqs_final_bs, color='0.6', lw=3.5, zorder=1)
    ax.plot(x, freqs_fi, color='steelblue', lw=1.3, zorder=2)
    for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
    ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
    ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0], x[-1]); ax.set_ylim(0, ymax)
    ax.set_title(f'Band-structure convergence — iteration {fi}/{len(theta_history)-1}')
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
    plt.tight_layout()
    images_bs.append(fig_to_image(fig))
    plt.close(fig)

imageio.mimsave('band_structure_convergence.gif', images_bs, fps=8, loop=0)
del images_bs; gc.collect()
print(f"Saved band_structure_convergence.gif ({len(frame_idx)} frames)")

# --- Boundary-seeking check ------------------------------------------------
zone_bnd = B_recip @ np.array([0.5, 0.5, 0.5])
f_fit   = bandstructure_freqs(K_LAB_FIT,   zone_bnd[None])[0]
f_prior = bandstructure_freqs(K_LAB_PRIOR, zone_bnd[None])[0]
ratio = f_fit/np.maximum(f_prior, 1e-12)
print(f"\nZone-boundary frequencies (THz):")
print(f"  prior: {f_prior.round(4)}")
print(f"  fit:   {f_fit.round(4)}")
print(f"  ratio: {ratio.round(3)}")
if ratio.min() < 0.1 or ratio.max() > 10:
    print("\n*** WARNING: a branch moved by more than an order of magnitude from the")
    print("    prior. That is the signature of the optimizer reaching a boundary")
    print("    rather than an interior optimum. Check the sloppiness spectrum and")
    print("    consider raising LAMBDA_PRIOR. ***")
else:
    print("\nAll branches within an order of magnitude of the prior: no boundary-seeking.")

In [ ]:
def fit_stats(K_lab, q_arr, G_arr, I_arr, scale=1.0):
    """R-factor AND the linear correlation coefficient.

    R alone is a poor summary once observations can be negative: Σ|I_obs| in the
    denominator is inflated by the magnitude of negatives, and R → 1 is exactly
    what a model predicting ~0 produces -- so R cannot tell 'poor fit' from
    'no fit'. CC separates those cases and is the statistic the diffuse-scattering
    literature reports.
    """
    Imodel = scale*diffuse_intensity_batch(K_lab, q_arr[:,0], q_arr[:,1], q_arr[:,2],
                                           G_arr=np.asarray(G_arr))
    R  = np.sum(np.abs(Imodel - I_arr))/np.sum(np.abs(I_arr))
    CC = np.corrcoef(Imodel, I_arr)[0,1] if len(I_arr) > 2 else np.nan
    return R, CC, Imodel

R_fit,  CC_fit,  I_model_fit  = fit_stats(K_LAB_FIT, q_fit,  G_fit,  I_fit,  S_FIT)
R_free, CC_free, I_model_test = fit_stats(K_LAB_FIT, q_test, G_test, I_test, S_FIT)
print(f"fitting set:  R = {R_fit:.4f}   CC = {CC_fit:.4f}")
print(f"held out:     R = {R_free:.4f}   CC = {CC_free:.4f}")
if len(hkl_halo):
    Rh, CCh, _ = fit_stats(K_LAB_FIT, hkl_halo, G_fit[is_halo], I_halo, S_FIT)
    print(f"halo bins:    R = {Rh:.4f}   CC = {CCh:.4f}")
print("\nR → 1 together with CC → 0 means the model is predicting essentially")
print("nothing; R → 1 with CC substantial means a scale or shape mismatch instead.")

# --- Per-resolution-shell breakdown: where does the model stop working? ----
q_abs = np.linalg.norm(q_test @ B_recip.T, axis=1)
d_test = 2*np.pi/np.maximum(q_abs, 1e-9)
edges = np.quantile(d_test, np.linspace(0, 1, 7))
print(f"\nHeld-out CC by resolution shell:")
shell_d, shell_cc = [], []
for b in range(len(edges)-1):
    m = (d_test >= edges[b]) & (d_test <= edges[b+1])
    if m.sum() < 20: continue
    cc = np.corrcoef(I_model_test[m], I_test[m])[0,1]
    shell_d.append(0.5*(edges[b]+edges[b+1])); shell_cc.append(cc)
    print(f"  {edges[b]:5.2f}–{edges[b+1]:5.2f} Å  ({m.sum():5d} pts):  CC = {cc:+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15,4.5))
for ax, Im, Io, title in [(axes[0], I_model_fit, I_fit, 'fitting set'),
                          (axes[1], I_model_test, I_test, 'held-out set')]:
    ax.scatter(Io, Im, s=6, alpha=0.35)
    lim = [min(Io.min(), Im.min())*1.05, max(Io.max(), Im.max())*1.05]
    ax.plot(lim, lim, 'k--', lw=1); ax.axhline(0, color='0.7', lw=0.6); ax.axvline(0, color='0.7', lw=0.6)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('$I_{obs}$ (experimental)'); ax.set_ylabel('$I_{model}$'); ax.set_title(title)
axes[2].plot(shell_d, shell_cc, 'o-', color='darkgreen')
axes[2].axhline(0, color='k', lw=0.8, ls='--')
axes[2].invert_xaxis(); axes[2].set_xlabel('resolution $d$ (Å)'); axes[2].set_ylabel('held-out CC')
axes[2].set_title('Where the model stops working')
plt.tight_layout(); plt.savefig('refinement_fit_quality.png', dpi=150); plt.show()

## How Many Parameters Does the Data Determine?

126 free parameters, and no reason to believe the data constrains 126 independent
combinations of them. The Gauss-Newton Hessian $J^{\mathsf T}J$, built from the
weighted Jacobian $J_{ij}=\sqrt{w_i}\,\partial I_i/\partial\theta_j$ at the
optimum, answers this directly: the spectrum in a model of this kind typically
falls off close to geometrically over many decades, the signature of a *sloppy*
model in Sethna's sense.

Three things this buys, all of which bear on the questions this notebook exists
to answer:

* $n_{\rm eff}$, the number of parameter **combinations** the data pins down. The
  rest are set by the prior, and the fitted $K$ should be reported saying so
  rather than quoted as 126 measured numbers.
* A principled `LAMBDA_PRIOR`: place it at the spectrum's noise floor, where it
  controls everything the data does not and nothing it does.
* An explanation for the initial-guess sensitivity. In the sloppy directions the
  answer *is* the prior, so a fit's dependence on its starting point is expected
  and quantifiable rather than mysterious — which is exactly why the prior is
  built from contact geometry instead of being an arbitrary isotropic matrix.

In [ ]:
# --- How many parameters does the data actually determine? -----------------
# The Sethna sloppiness diagnostic. Build the weighted Jacobian
# J_ij = sqrt(w_i) dI_i/dθ_j at the optimum and look at the spectrum of J^T J.
# For a model like this expect a handful of stiff directions out of 126, with
# the eigenvalues falling off geometrically over many decades. What this buys:
#   * n_eff -- the number of parameter COMBINATIONS the data constrains;
#   * a principled LAMBDA_PRIOR: place it at the noise floor of the spectrum,
#     so it controls everything the data does not and nothing it does;
#   * an honest way to report the result ("prior, plus data-determined
#     corrections along n_eff directions") instead of quoting 126 numbers as
#     though they were all measured.

def weighted_jacobian(theta, q_list, G_list, weights, scale=1.0, eps=1e-5):
    sw = np.sqrt(np.asarray(weights, float))
    q_list = np.asarray(q_list, float); G_arr = np.asarray(G_list)
    qb = (q_list[:,0:1]*B_recip[:,0] + q_list[:,1:2]*B_recip[:,1]
          + q_list[:,2:3]*B_recip[:,2])
    def I_of(th):
        _, K_lab = build_K(theta_to_Lmap(th))
        D = dynamical_matrix_batch(qb, unique, K_lab)
        V = np.linalg.solve(D, G_arr[:,:,None])[:,:,0]
        return scale*np.real(np.sum(np.conj(G_arr)*V, axis=1))
    Jm = np.zeros((len(q_list), len(theta)))
    for j in range(len(theta)):
        tp = theta.copy(); tp[j] += eps
        tm = theta.copy(); tm[j] -= eps
        Jm[:, j] = sw*(I_of(tp) - I_of(tm))/(2*eps)
    return Jm

def report_sloppiness(theta, q_list, G_list, weights, scale=1.0,
                      noise_floor_frac=1e-6, fname='sloppiness_spectrum.png'):
    Jm = weighted_jacobian(theta, q_list, G_list, weights, scale)
    ev = np.linalg.eigvalsh(Jm.T @ Jm)[::-1]
    ev = np.clip(ev, 1e-300, None)
    n_eff = int(np.sum(ev > noise_floor_frac*ev[0]))
    print(f"Gauss-Newton spectrum over {len(ev)} parameters:")
    print(f"  λ_max = {ev[0]:.4e}   λ_min = {ev[-1]:.4e}   "
          f"dynamic range = {ev[0]/ev[-1]:.2e}")
    print(f"  directions above {noise_floor_frac:g}·λ_max: {n_eff} / {len(ev)}")
    print(f"  -> the remaining {len(ev)-n_eff} directions are set by the prior, "
          f"not by the data. Say so when reporting K.")
    fig, ax = plt.subplots(figsize=(7,4))
    ax.semilogy(np.arange(1, len(ev)+1), ev/ev[0], 'o-', ms=3)
    ax.axhline(noise_floor_frac, color='r', ls='--', lw=1,
               label=f'noise floor ({noise_floor_frac:g})')
    ax.axvline(n_eff+0.5, color='k', ls=':', lw=1, label=f'$n_{{eff}}$ = {n_eff}')
    ax.set_xlabel('eigenvalue index'); ax.set_ylabel(r'$\lambda_i/\lambda_1$')
    ax.set_title('Sloppiness: Gauss-Newton eigenvalue spectrum')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()
    return ev, n_eff


In [ ]:
ev_spec, n_eff = report_sloppiness(theta_fit, q_fit, G_fit, w_fit, scale=S_STAR)

print(f"\nThe fit determines ~{n_eff} of {N_PARAMS} parameter combinations; the")
print(f"remaining {N_PARAMS-n_eff} are the prior showing through.")
print(f"current LAMBDA_PRIOR = {LAMBDA_PRIOR:g}   "
      f"λ_max = {ev_spec[0]:.3e}   λ_max·1e-6 = {ev_spec[0]*1e-6:.3e}")

## Experimental vs. Fitted Diffuse Scattering

The experimental $l\approx0$ plane next to the model's prediction on **exactly
the same** $(h,k)$ grid points. The refinement only ever saw the fitting points;
this compares the two everywhere in the plane.

Because $G(\mathbf q)$ is now summed over ~10³ atoms rather than ~3×10⁵ density
voxels, evaluating a full plane is cheap and `PLANE_STRIDE` can be small — this
cell used to be dominated by the transform.

The intensity panels are $\log_{10}(1+I)$; the residual panels are not
log-transformed. The absolute residual is dominated by the brightest halo pixels
even when the fit is good elsewhere, which is what the relative panel corrects
for. NaN voxels (outside the measured region) are masked from all four panels.

One caveat on reading this figure: both panels use `diffuse_intensity_batch`,
the *same* code path asserted earlier against the single-point solve the
objective uses. Without that assertion there would be no guarantee the plotted
model is the fitted model.

In [ ]:
PLANE_STRIDE = 3   # the atomic transform makes a full plane affordable

h_pts_cmp0 = h_pts_bg0[::PLANE_STRIDE]
k_pts_cmp0 = k_pts_bg0[::PLANE_STRIDE]
I_bg0_cmp  = I_bg0[::PLANE_STRIDE, ::PLANE_STRIDE]
print(f"Comparison plane: {len(h_pts_cmp0)}×{len(k_pts_cmp0)} points "
      f"(stride={PLANE_STRIDE}, full plane {len(h_pts_bg0)}×{len(k_pts_bg0)})")

def predicted_plane(K_lab, h_pts, k_pts, l_layer, scale=1.0):
    HH, KK = np.meshgrid(h_pts, k_pts, indexing='ij')
    h_flat, k_flat = HH.ravel(), KK.ravel()
    l_flat = np.full_like(h_flat, l_layer)
    return (scale*diffuse_intensity_batch(K_lab, h_flat, k_flat, l_flat)).reshape(HH.shape)

I_fit_map0 = predicted_plane(K_LAB_FIT, h_pts_cmp0, k_pts_cmp0, l_layer_0, scale=S_FIT)
exp_mask0  = np.isfinite(I_bg0_cmp)

def plot_diffuse_comparison(h_pts, k_pts, I_exp, I_fit, mask, fname, rel_clip=1.0):
    I_exp_m = np.where(mask, I_exp, np.nan)
    I_fit_m = np.where(mask, I_fit, np.nan)
    pos_vals = np.concatenate([I_exp_m[mask & (I_exp_m > 0)], I_fit_m[mask & (I_fit_m > 0)]])
    vmax = np.percentile(pos_vals, 97) if len(pos_vals) else 1.0
    resid = np.where(mask, I_fit_m - I_exp_m, np.nan)
    rmax = np.nanpercentile(np.abs(resid), 99) if np.isfinite(resid).any() else 1.0
    rel = resid/np.maximum(np.abs(I_exp_m), 0.02*vmax)
    extent = [h_pts[0], h_pts[-1], k_pts[0], k_pts[-1]]

    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    im0 = axes[0,0].imshow((np.log1p(np.clip(I_exp_m, 0, vmax))/np.log(10)).T, origin='lower',
                           cmap='inferno', extent=extent, aspect='auto')
    axes[0,0].set_title('Experimental (CXIDB 128)')
    plt.colorbar(im0, ax=axes[0,0], label=r'$\log_{10}(1+I)$')
    im1 = axes[0,1].imshow((np.log1p(np.clip(I_fit_m, 0, vmax))/np.log(10)).T, origin='lower',
                           cmap='inferno', extent=extent, aspect='auto')
    axes[0,1].set_title('Fitted $K$')
    plt.colorbar(im1, ax=axes[0,1], label=r'$\log_{10}(1+I)$')
    im2 = axes[1,0].imshow(resid.T, origin='lower', cmap='RdBu_r',
                           extent=extent, aspect='auto', vmin=-rmax, vmax=rmax)
    axes[1,0].set_title('Absolute residual (fit − experimental)')
    plt.colorbar(im2, ax=axes[1,0], label=r'$\Delta I$ (raw units, not log)')
    im3 = axes[1,1].imshow(np.clip(rel, -rel_clip, rel_clip).T, origin='lower', cmap='RdBu_r',
                           extent=extent, aspect='auto', vmin=-rel_clip, vmax=rel_clip)
    axes[1,1].set_title(f'Relative residual, clipped to ±{100*rel_clip:.0f}%')
    plt.colorbar(im3, ax=axes[1,1], label=r'$\Delta I / I_{\rm exp}$')
    for ax in axes.flat:
        ax.set_xlabel('h'); ax.set_ylabel('k')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

    ok = np.isfinite(resid) & np.isfinite(I_exp_m)
    cc = np.corrcoef(I_fit_m[ok], I_exp_m[ok])[0,1]
    print(f"Saved {fname}")
    print(f"Plane CC (fit vs. experimental): {cc:.4f}")
    print(f"Absolute residual RMS: {np.sqrt(np.nanmean(resid**2)):.4g}  "
          f"(I_exp RMS: {np.sqrt(np.nanmean(I_exp_m**2)):.4g})")
    print(f"Fraction of the experimental variance explained: {cc**2:.1%}")

plot_diffuse_comparison(h_pts_cmp0, k_pts_cmp0, I_bg0_cmp, I_fit_map0, exp_mask0,
                        'diffuse_map_experimental_vs_fit_hk0.png')

## Predicted vs. Deposited Atomic Displacement Parameters

$\Sigma=\langle(\Omega,v)(\Omega,v)^{\mathsf T}\rangle$ from integrating
$D(\mathbf q)^{-1}$ over the Brillouin zone; projecting onto an atom via
$\delta\mathbf r=\mathbf v+\boldsymbol\Omega\times(\mathbf r-\mathbf r_{\rm cm})$
gives its full anisotropic tensor $U=J_r\Sigma J_r^{\mathsf T}$, with
$B=\tfrac{8\pi^2}{3}\mathrm{Tr}(U)$.

**This is the independent check, and nothing above used it.** $K$ was fixed
absolutely by the unit argument, not by matching a deposited $B$. So three things
are genuine predictions here, and should be read in this order:

1. **The mean.** The prediction should come out **below** the deposited mean,
   because the deposited $B$ also absorbs internal motion, side-chain disorder
   and substitutional heterogeneity that a single-rigid-body lattice model cannot
   and should not reproduce. A prediction *above* the deposited mean means $K$ is
   too soft; a prediction at exactly the deposited mean would mean the lattice
   accounts for all of the disorder, which is not physical.
2. **The per-atom shape.** How $B$ varies across the molecule. Note the floor
   here: a rigid body predicts $B$ growing with distance from the centre of mass,
   which correlates with real $B$ at $r\approx0.4$–$0.6$ from geometry alone. A
   correlation in that range is therefore *not* evidence the dynamics are right.
3. **The anisotropy.** The shape of each displacement ellipsoid, which geometry
   alone does not give you, and which is the most informative of the three.

In [ ]:
def bz_covariance(K_lab, n_grid=BZ_NGRID, offset=True, eig_floor=None, chunk=None):
    """Σ = <(Ω,v)(Ω,v)^T>, from integrating D(q)^-1 over the Brillouin zone.

    Three choices matter here and all three were wrong in an earlier version:

    * the FULL COMPLEX Hermitian D is used (D.real is a different matrix and
      shifts trace Σ by ~20% on this geometry);
    * the grid is OFFSET (Monkhorst-Pack style) so it never lands on Γ. The
      acoustic contribution goes as ∫d³q/q² -- convergent in 3D but badly
      undersampled by a coarse Γ-inclusive grid, and Σ is the headline ADP
      prediction, so this is not a detail;
    * the eigenvalue floor is ABSOLUTE, not relative to ev.max() at each q.
      With a strongly anisotropic K, a genuine small acoustic eigenvalue near Γ
      can otherwise be zeroed just because ev.max() happens to be large there.
    """
    chunk = chunk or CHUNK_EVAL
    g = (np.arange(n_grid) + (0.5 if offset else 0.0))/n_grid
    Ig, Jg, Kg = np.meshgrid(g, g, g, indexing='ij')
    q_batch = np.column_stack([Ig.ravel(), Jg.ravel(), Kg.ravel()]) @ B_recip.T
    N = len(q_batch)

    if eig_floor is None:
        Dref = dynamical_matrix_batch(q_batch[:min(N, 2000)], unique, K_lab)
        eig_floor = 1e-9 * mass_weighted_eigs(Dref).max()

    Sigma = np.zeros((6,6))
    for s0 in range(0, N, chunk):
        sl = slice(s0, min(s0+chunk, N))
        Db = dynamical_matrix_batch(q_batch[sl], unique, K_lab)
        Dw = np.einsum('ij,njk,lk->nil', Msq_inv, Db, Msq_inv)
        ev, evec = np.linalg.eigh(Dw)
        e_lab = np.einsum('ij,njk->nik', Msq_inv.T, evec)
        mask  = ev > eig_floor
        coeff = np.where(mask, 1.0/np.where(mask, ev, 1.0), 0.0)
        Sigma += np.real(np.einsum('ns,nis,njs->ij', coeff, e_lab, e_lab.conj()))
    return Sigma/N

def atom_adp_tensors(Sigma, positions):
    """Full anisotropic displacement tensor U (Å², 3x3) per atom. Vectorized."""
    d = np.asarray(positions) - r_cm_at
    N = len(d)
    Jr = np.zeros((N, 3, 6))
    Jr[:,0,1] =  d[:,2]; Jr[:,0,2] = -d[:,1]
    Jr[:,1,0] = -d[:,2]; Jr[:,1,2] =  d[:,0]
    Jr[:,2,0] =  d[:,1]; Jr[:,2,1] = -d[:,0]
    Jr[:,0,3] = Jr[:,1,4] = Jr[:,2,5] = 1.0
    return np.einsum('nia,ab,njb->nij', Jr, Sigma, Jr)

def isotropic_B(U):
    return (8*np.pi**2/3) * np.trace(U, axis1=1, axis2=2)

def mean_predicted_B(K_lab, n_grid=BZ_NGRID):
    return isotropic_B(atom_adp_tensors(bz_covariance(K_lab, n_grid=n_grid), apos)).mean()

def check_bz_convergence(K_lab, grids=(8, 12, 16, BZ_NGRID)):
    """Σ is the headline ADP prediction -- never report it without this."""
    print("BZ-grid convergence of the predicted mean B:")
    prev = None
    for n in grids:
        B = mean_predicted_B(K_lab, n_grid=n)
        delta = '' if prev is None else f'   Δ = {100*(B-prev)/prev:+.2f}%'
        print(f"  n_grid={n:3d}³ ({n**3:6d} q-points):  mean B = {B:8.3f} Å²{delta}")
        prev = B


Sigma_fit = bz_covariance(K_LAB_FIT)
U_fit = atom_adp_tensors(Sigma_fit, apos)
B_fit = isotropic_B(U_fit)

check_bz_convergence(K_LAB_FIT)
print()
print(f"PREDICTED mean B (fitted K):  {B_fit.mean():8.3f} Å²")
print(f"Deposited mean B (6o2h):      {b_exp.mean():8.3f} Å²")
ratio = B_fit.mean()/b_exp.mean()
print(f"ratio predicted/deposited:    {ratio:8.3f}")
if ratio > 1.0:
    print("\n  > 1: the model predicts MORE displacement than was refined from the")
    print("  Bragg data, so the fitted contacts are too soft. Something is wrong --")
    print("  check s*, the resolution range, and the sloppiness spectrum.")
else:
    print(f"\n  < 1, as it should be: the lattice model accounts for {100*ratio:.0f}% of the")
    print("  deposited B, with the remainder from internal and substitutional")
    print("  disorder outside this model. That split is the result, not a defect.")

In [ ]:
res_ids = []
for ch in st[0]:
    for res in ch:
        for atom in res:
            res_ids.append((ch.name, res.seqid.num))
res_index = {}
res_inverse = np.empty(len(res_ids), dtype=int)
for i, key_ in enumerate(res_ids):
    res_inverse[i] = res_index.setdefault(key_, len(res_index))
n_res = len(res_index)

def per_residue_mean(v):
    out = np.zeros(n_res); counts = np.zeros(n_res)
    np.add.at(out, res_inverse, v); np.add.at(counts, res_inverse, 1)
    return out/counts

B_exp_res, B_fit_res = per_residue_mean(b_exp), per_residue_mean(B_fit)

# Geometry-only baseline: what correlation does a rigid body give from the lever
# arm alone, with no dynamical content at all? Anything at or below this is not
# evidence the fitted dynamics are right.
lever = np.linalg.norm(apos - r_cm_at, axis=1)**2
r_geom = np.corrcoef(b_exp, lever)[0,1]
r_fit  = np.corrcoef(b_exp, B_fit)[0,1]

fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
ax = axes[0]
ax.scatter(b_exp, B_fit, s=6, alpha=0.4)
lim = [0, max(b_exp.max(), B_fit.max())*1.05]
ax.plot(lim, lim, 'k--', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('$B$ — deposited 6o2h (Å²)'); ax.set_ylabel('$B$ — fitted $K$ (Å²)')
ax.set_title(f'Per-atom isotropic $B$   (r = {r_fit:.3f})')

ax = axes[1]
ax.plot(B_exp_res, label='deposited', lw=1.3)
ax.plot(B_fit_res, label='fitted $K$', lw=1.3, alpha=0.8)
ax.set_xlabel('residue index'); ax.set_ylabel('$B$ (Å²)')
ax.set_title('Per-residue mean $B$'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('adp_comparison.png', dpi=150); plt.show()

print(f"Pearson r, per-atom B (deposited vs. fitted):     {r_fit:.3f}")
print(f"Pearson r, deposited B vs. |r - r_cm|² alone:     {r_geom:.3f}")
print(f"  -> the model beats the geometry-only baseline by {r_fit - r_geom:+.3f}.")
print("  A fitted correlation at or below the baseline means the dynamics are")
print("  contributing nothing beyond the rigid-body lever arm.")

### Anisotropic Comparison

The isotropic trace discards the shape of each displacement ellipsoid. Where 6o2h
carries ANISOU records the full tensor comparison checks whether the refinement
recovers that shape: the six independent components of $U$ against each other,
and each atom's anisotropy ratio (largest/smallest principal displacement).

This is the most informative of the three ADP tests, because unlike the isotropic
trace it cannot be reproduced by the lever-arm geometry alone. A model that
matches on the trace but gets the anisotropy wrong would look fine above and fail
here.

In [ ]:
def anisotropy_ratio(U):
    ev = np.linalg.eigvalsh(U)
    return ev[:, -1] / np.clip(ev[:, 0], 1e-12, None)

comp_idx   = [(0,0),(1,1),(2,2),(0,1),(0,2),(1,2)]
comp_names = ['$U_{11}$','$U_{22}$','$U_{33}$','$U_{12}$','$U_{13}$','$U_{23}$']

if has_aniso.mean() > 0.3:
    fig, axes = plt.subplots(1, 2, figsize=(11,4.5))

    ax = axes[0]
    for (i,j), name in zip(comp_idx, comp_names):
        ax.scatter(u_exp[has_aniso,i,j], U_fit[has_aniso,i,j], s=5, alpha=0.4, label=name)
    lim = [min(u_exp[has_aniso].min(), U_fit[has_aniso].min()),
           max(u_exp[has_aniso].max(), U_fit[has_aniso].max())]
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlabel('$U$ component — deposited 6o2h (Å²)'); ax.set_ylabel('$U$ component — fitted $K$ (Å²)')
    ax.set_title('Anisotropic tensor components: deposited vs. fitted')
    ax.legend(fontsize=7, ncol=2)

    ax = axes[1]
    aniso_exp = anisotropy_ratio(u_exp[has_aniso])
    aniso_fit = anisotropy_ratio(U_fit[has_aniso])
    ax.scatter(aniso_exp, aniso_fit, s=6, alpha=0.4)
    lim2 = [1, max(aniso_exp.max(), aniso_fit.max())*1.05]
    ax.plot(lim2, lim2, 'k--', lw=1)
    ax.set_xlabel('anisotropy ratio — deposited'); ax.set_ylabel('anisotropy ratio — fitted $K$')
    ax.set_title('Displacement-ellipsoid anisotropy: deposited vs. fitted')

    plt.tight_layout(); plt.savefig('adp_anisotropic_comparison.png', dpi=150); plt.show()
else:
    print(f"Only {has_aniso.mean():.1%} of atoms in 6o2h carry ANISOU records; "
          f"skipping the anisotropic comparison. Isotropic B-factors are "
          f"compared above regardless.")


## Phonon Mode Visualization

Each phonon mode at a chosen $\mathbf q$-point is rendered as a looping GIF
of the rigid-body motion, using the model **fitted to the experimental
data**. $|\Omega|$ and $|v|$ are the rotational and translational norms of
the mass-weighted eigenvector (normalized so
$\Omega^{\mathsf T}J\Omega+v^{\mathsf T}mv=1$); their ratio gives the
rotational kinetic-energy fraction, used below to label each mode
librational, translational, or mixed. Displacement amplitude is defined as
the largest per-atom displacement in Å, not the center-of-mass displacement.


In [ ]:
Q_VIZ     = np.array([0.5, 0.25, 0.0])  # fractional — zone-boundary, off-axis
AMPLITUDE = 3.0    # Å — maximum per-atom displacement
N_FRAMES  = 30
GIF_FPS   = 12

q_viz  = B_recip @ Q_VIZ
D_viz  = dynamical_matrix(q_viz, unique, K_LAB_FIT)
# Complex Hermitian, as everywhere else -- Re(D) has different eigenvalues, and
# the acoustic (smallest) ones are the worst affected, so a .real here would
# animate a different set of modes from the ones that were fit.
Dw_viz = np.einsum('ij,jk,lk->il', Msq_inv, D_viz, Msq_inv)
ev_v, evec_v = np.linalg.eigh(Dw_viz)
ev_v      = np.maximum(ev_v, 0)
freqs_viz = np.sqrt(ev_v) * freq_unit
evecs_lab = Msq_inv.T @ evec_v          # complex, one column per mode

# A phonon eigenvector at general q is genuinely complex: the physical motion is
# Re(e e^{i(q·R - ωt)}), so different components lead each other in phase. For a
# single-cell animation we rotate each mode by the global phase that makes it as
# real as possible (which is exact for a standing mode and a good approximation
# otherwise) and record how much amplitude the residual imaginary part carries.
_phase = np.exp(-1j*np.angle(evecs_lab[np.abs(evecs_lab).argmax(axis=0),
                                       np.arange(6)]))
evecs_lab = evecs_lab * _phase[None, :]
_im_frac = np.linalg.norm(evecs_lab.imag, axis=0)/np.linalg.norm(evecs_lab, axis=0)
print(f"Residual out-of-phase amplitude per mode: {_im_frac.round(3)}")
print("  (0 = a pure standing mode the single-cell animation represents exactly;")
print("   large values mean the components genuinely lead each other in phase.)")

print(f"Modes at q={Q_VIZ}:")
hdr = f"{'Mode':>5} {'Freq (THz)':>14} {'|Ω|':>9} {'|v|':>9}  {'rot KE %':>9}  character"
print(hdr); print('-'*len(hdr))
for s in range(6):
    Om = evecs_lab[:3, s].real; v = evecs_lab[3:, s].real
    rot_KE, trans_KE = float(Om @ J @ Om), float(m_total * (v@v))
    rot_pct = 100 * rot_KE / max(rot_KE + trans_KE, 1e-30)
    char = 'librational' if rot_pct > 60 else ('translational' if rot_pct < 40 else 'mixed')
    print(f"  {s:3d}  {freqs_viz[s]:14.6f}  {np.linalg.norm(Om):9.4f}  "
          f"{np.linalg.norm(v):9.4f}  {rot_pct:9.1f}%  {char}")


In [ ]:
# Molecular isosurface: marching cubes on the calculated model density
# (rho_model, built earlier for visualization only -- the molecular transform
# used for G is summed over atoms, not read off this grid), downsampled
# for speed, keeping only the largest connected component so periodic-image
# and solvent-void fragments are discarded, then Lambert-shaded per face so
# shape and roughness stay visible even at partial transparency.
base_pos = apos.copy()
pts_cm   = base_pos - r_cm_at

MC_DS = 0.5
rho_mc = nd_zoom(rho_model.astype(np.float32), MC_DS, order=1)
nu_mc, nv_mc, nw_mc = rho_mc.shape
_iso_level = rho_mc.max() * 0.05
mc_verts_grid, mc_faces_raw, _normals, _vals = marching_cubes(rho_mc, level=_iso_level)
_frac = mc_verts_grid / np.array([nu_mc, nv_mc, nw_mc])
mc_verts_raw = (A_orth @ _frac.T).T

def _largest_component(verts, faces):
    n = len(verts)
    idx_i = np.concatenate([faces[:,0], faces[:,1], faces[:,2]])
    idx_j = np.concatenate([faces[:,1], faces[:,2], faces[:,0]])
    adj   = csr_matrix((np.ones(len(idx_i), dtype=np.int8), (idx_i, idx_j)), shape=(n, n))
    _, labels = sc_connected_components(adj, directed=False)
    keep_label = np.bincount(labels).argmax()
    keep = np.where(labels == keep_label)[0]
    remap = np.full(n, -1, dtype=int); remap[keep] = np.arange(len(keep))
    face_ok = (labels[faces] == keep_label).all(axis=1)
    return verts[keep], remap[faces[face_ok]]

mc_verts_cart, mc_faces = _largest_component(mc_verts_raw, mc_faces_raw)
mc_verts_cm = mc_verts_cart - r_cm_at
print(f"Isosurface: {len(mc_verts_cm)} verts, {len(mc_faces)} tris "
      f"(largest connected component, level={_iso_level:.3f} e/Å³)")

_LIGHT = np.array([0.5, 0.8, 1.0]); _LIGHT /= np.linalg.norm(_LIGHT)

def _shade(tri_array, hex_color, alpha, ambient=0.35):
    v0, v1, v2 = tri_array[:,0], tri_array[:,1], tri_array[:,2]
    n = np.cross(v1 - v0, v2 - v0)
    mag = np.linalg.norm(n, axis=1, keepdims=True)
    n /= np.where(mag > 1e-12, mag, 1.0)
    intensity = ambient + (1 - ambient) * np.abs(n @ _LIGHT)
    r = int(hex_color[1:3], 16) / 255
    g = int(hex_color[3:5], 16) / 255
    b = int(hex_color[5:7], 16) / 255
    return np.column_stack([intensity*r, intensity*g, intensity*b, np.full(len(tri_array), alpha)])

def displaced_mc(mode_idx, scale):
    Om_raw = evecs_lab[:3, mode_idx].real.copy()
    v_raw  = evecs_lab[3:, mode_idx].real.copy()
    norm_vec = np.sqrt(Om_raw @ Om_raw + v_raw @ v_raw)
    if norm_vec < 1e-10:
        return mc_verts_cm + r_cm_at
    Om_u, v_u = Om_raw/norm_vec, v_raw/norm_vec
    delta_unit = v_u[None,:] + np.cross(Om_u[None,:], pts_cm)
    max_d = np.linalg.norm(delta_unit, axis=1).max()
    if max_d < 1e-12:
        return mc_verts_cm + r_cm_at
    fac = scale / max_d
    Om_s, v_s = fac*Om_u, fac*v_u
    if abs(fac)*np.linalg.norm(Om_u) > 1e-10:
        return Rot.from_rotvec(Om_s).apply(mc_verts_cm) + r_cm_at + v_s
    return mc_verts_cm + r_cm_at + v_s

eq_bbox_min = mc_verts_cart.min(axis=0)
eq_bbox_max = mc_verts_cart.max(axis=0)


In [ ]:
MODE_COLORS = ['#4e9de0', '#e05c5c', '#4fba74', '#e0b14e', '#a56be0', '#e07e4e']
SURF_ALPHA  = 0.65
ELEV, AZIM  = 20, -60

def make_mode_gif(mode_idx, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS, fname=None, figsize=(6,5)):
    """Render one phonon mode as a looping GIF and save it to disk (no
    in-notebook display, and memory is freed after each mode)."""
    if fname is None:
        qstr = '_'.join(f'{x:.2f}' for x in Q_VIZ)
        fname = f'mode_{mode_idx}_q{qstr}.gif'
    color  = MODE_COLORS[mode_idx % len(MODE_COLORS)]
    scales = amplitude * np.sin(2*np.pi*np.arange(n_frames)/n_frames)
    pad = amplitude * 2.5
    xlim = (eq_bbox_min[0]-pad, eq_bbox_max[0]+pad)
    ylim = (eq_bbox_min[1]-pad, eq_bbox_max[1]+pad)
    zlim = (eq_bbox_min[2]-pad, eq_bbox_max[2]+pad)

    Om_r = evecs_lab[:3, mode_idx].real
    rot_KE   = float(Om_r @ J @ Om_r)
    trans_KE = m_total * float(np.linalg.norm(evecs_lab[3:, mode_idx].real)**2)
    rot_pct  = 100*rot_KE/(rot_KE+trans_KE+1e-30)

    images = []
    for sc in scales:
        fig = plt.figure(figsize=figsize, facecolor='#0d1117')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            verts = displaced_mc(mode_idx, sc)
            tris  = verts[mc_faces]
            rgba  = _shade(tris, color, SURF_ALPHA)
            poly  = Poly3DCollection(tris, facecolors=rgba, linewidths=0, edgecolor='none')
            ax.add_collection3d(poly)
            ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
            ax.set_axis_off(); ax.view_init(elev=ELEV, azim=AZIM)
            ax.set_title(f'Mode {mode_idx}   {freqs_viz[mode_idx]:.4f} THz\n'
                         f'q={list(Q_VIZ)}   rot {rot_pct:.0f}%  trans {100-rot_pct:.0f}%',
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)

    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"Generating surface GIFs at q={Q_VIZ} (amplitude={AMPLITUDE} Å max per-atom)...")
for s in range(6):
    print(f"Mode {s}: {freqs_viz[s]:.6f} THz", end="  ... ")
    try:
        path = make_mode_gif(s)
        print(f"saved -> {path}")
    except Exception as e:
        print(f"FAILED: {e}")
    gc.collect()
print("Load any GIF with: display(IPImage(filename='mode_N_q....gif'))")


In [ ]:
# Supercell wave: each cell (i,j,k) is a rigid copy displaced by
# amplitude * cos(2*pi*q_frac.[i,j,k] + phase). The central cell is
# colored distinctly from its neighbors so the propagating wave pattern
# can be read off against a fixed reference point.
SUPER_N1, SUPER_N2, SUPER_N3 = 3, 3, 1   # a full 3x3x3 renders 27 isosurfaces per
                                          # frame and can exhaust notebook-kernel memory;
                                          # 3x3x1 already shows the in-plane wave pattern
MODE_SUPER   = 0
SUPER_COLOR  = '#4e9de0'
SUPER_ALPHA  = 0.38
CENTER_COLOR = '#e05c5c'
CENTER_ALPHA = 0.55

def make_supercell_gif(mode_idx=MODE_SUPER, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS,
                       n1=SUPER_N1, n2=SUPER_N2, n3=SUPER_N3, fname=None, figsize=(10,9)):
    if fname is None:
        qstr = '_'.join(f'{x:.2f}' for x in Q_VIZ)
        fname = f'supercell_mode{mode_idx}_q{qstr}.gif'

    cells_ijk = [(i,j,k) for i in range(n1) for j in range(n2) for k in range(n3)]
    center_ijk = (n1//2, n2//2, n3//2)
    phase0 = np.array([2*np.pi*(Q_VIZ[0]*i + Q_VIZ[1]*j + Q_VIZ[2]*k) for i,j,k in cells_ijk])
    T_cell = np.array([i*a1 + j*a2 + k*a3 for i,j,k in cells_ijk])

    all_eq = np.vstack([mc_verts_cart + T for T in T_cell])
    pad = amplitude * 2.5
    xlim = (all_eq[:,0].min()-pad, all_eq[:,0].max()+pad)
    ylim = (all_eq[:,1].min()-pad, all_eq[:,1].max()+pad)
    zlim = (all_eq[:,2].min()-pad, all_eq[:,2].max()+pad)

    images = []
    for fi in range(n_frames):
        t = 2*np.pi*fi/n_frames
        fig = plt.figure(figsize=figsize, facecolor='#0d1117')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            for (i,j,k), T, ph0 in zip(cells_ijk, T_cell, phase0):
                sc = amplitude * np.cos(ph0 + t)
                verts = displaced_mc(mode_idx, sc) + T
                tris_v = verts[mc_faces]
                is_center = (i,j,k) == center_ijk
                color = CENTER_COLOR if is_center else SUPER_COLOR
                alpha = CENTER_ALPHA if is_center else SUPER_ALPHA
                rgba = _shade(tris_v, color, alpha)
                poly = Poly3DCollection(tris_v, facecolors=rgba, linewidths=0, edgecolor='none')
                ax.add_collection3d(poly)

            ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
            ax.set_axis_off(); ax.view_init(elev=25, azim=-55)
            ax.set_title(f'Supercell phonon — Mode {mode_idx}  {freqs_viz[mode_idx]:.4f} THz\n'
                         f'q={list(Q_VIZ)}   {n1}×{n2}×{n3} cells (center cell highlighted)',
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)

    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"Generating {SUPER_N1}×{SUPER_N2}×{SUPER_N3} supercell GIF for mode {MODE_SUPER}...")
try:
    sc_path = make_supercell_gif()
    print(f"Saved -> {sc_path}")
except Exception as e:
    print(f"Supercell GIF failed: {e}")
    import traceback; traceback.print_exc()


## Summary

| Output | File | Description |
|--------|------|-------------|
| Band structure (prior) | `band_structure_prior.png` | Phonon dispersion Γ–X–Y–Z–Γ before fitting |
| Resolution diagnostics | `resolution_snr_scan.png` | SNR, coverage and negative fraction vs. resolution |
| Bragg-distance diagnostics | `bragg_distance_diagnostic.png` | Negative fraction, mean $I$, and population vs. halo distance |
| Sampled q-points | `sampled_q_points.png` | 3D view plus slab overlays on the experimental map |
| Refinement fit quality | `refinement_fit_quality.png` | $I_{\rm obs}$ vs $I_{\rm model}$, plus held-out CC by resolution |
| Sloppiness spectrum | `sloppiness_spectrum.png` | Gauss-Newton eigenvalues; how many parameters the data determines |
| Band-structure convergence | `band_structure_convergence.gif` | Convergence toward the final fit |
| Experimental vs. fitted map | `diffuse_map_experimental_vs_fit_hk0.png` | Data, fit, and residuals |
| ADP comparison (isotropic) | `adp_comparison.png` | Per-atom and per-residue $B$, with a geometry-only baseline |
| ADP comparison (anisotropic) | `adp_anisotropic_comparison.png` | Full $U$ components and anisotropy ratio |
| Mode GIFs / supercell | `mode_N_q*.gif`, `supercell_mode*.gif` | Fitted-model phonon modes |

**Molecular transform.** $F$ and $L$ are summed directly over the deposited
atomic coordinates with IT92 form factors, referenced to the atomic centre of
mass — the same point the dynamical matrix uses. A periodic density grid gives
the wrong $F(\mathbf q)$ at the non-integer $(h,k,l)$ this pipeline runs on,
because a molecule crossing a cell boundary contributes with unequal phase
factors, and no roll of the grid can fix that for a protein whose contacts are
contiguous. Measured Bragg amplitudes enter as a smooth resolution-dependent
amplitude correction instead of through a hybrid density.

**Everything is complex.** $I=G^\dagger D^{-1}G$ has complex $G$ and complex
Hermitian $D$, with $\mathrm{Im}\,D_{\mathbf n}=\sin(\mathbf q\cdot\mathbf R_{\mathbf n})
(KA-(KA)^{\mathsf T})$ generically $O(1)$. Replacing either by its real part is a
different model, not an approximation. Cross-path assertions guarantee the
plotted and integrated model is the fitted one.

**Contact stiffness and prior.** Six *distinct* contacts ($P1$ has no point
symmetry, so they are not symmetry-related), each a free $6\times6$ Cholesky
factor in a frame anchored at the measured contact centroid. The prior is the
stiffness of $n$ isotropic point springs at the atom-pair midpoints, supplying the
contact-count scaling, the $\kappa_R/\kappa_T\sim\rho_g^2$ ratio, and the patch
anisotropy for free; refinement is regularized toward it rather than toward zero.

**Scale.** The $K$/scale degeneracy is removed from the parameter space by
refining the normalized shape of $K$ with $s$ profiled in closed form, then closed
with units ($s\equiv1$, since $I_{\rm model}$ is in electrons² per unit cell).
**No ADP information enters the fit**, so the predicted $B$-factors and ANISOU
tensors remain an independent check — and the predicted mean $B$ is expected to
fall *below* the deposited mean, since the latter also contains disorder outside
this model.

**Data selection.** A halo profile (shell × sector averages around every
reciprocal-lattice point, which beats the noise down by $\sim\sqrt N$ and reads
out the acoustic limit) plus a stratified mid-zone sample, both on a resolution
range chosen for model validity rather than by an SNR heuristic. Negative
intensities are kept — filtering on the sign of the fitted quantity would bias the
result — and the effective sample size is reported everywhere so a nominal count
cannot disguise a fit resting on few observations.

**Refinement.** L-BFGS-B with an analytic gradient verified against finite
differences, fully vectorized. Convergence is asserted. $R$ **and** $CC$ are
reported, per resolution shell, and the Gauss-Newton spectrum states how many of
the 126 parameters the data actually determines.